# Complete Transit Nodes Processing Pipeline

This notebook integrates the complete workflow:

## Part 1: H3 Hexagon Processing (Steps 1-4)
1. Load transit nodes from CSV
2. Assign H3 hexagonal indices
3. Group hexagons by 120m proximity (edge-to-edge)
4. Geocode addresses (optional)
5. Export grouped hexagons

## Part 2: Demand Data Processing (Steps 5-8)
5. Load grouped hexagons
6. Tag with spatial information (metro areas, districts)
7. Add daily demand from regional transport models
8. Create aggregated hub groups with demand statistics

## Part 3: Influence Area Processing (Steps 9-11) [Optional]
9. Create buffer zones (0-600m, 600-1000m, 1000-1200m)
10. Calculate population and employment from TAZ data
11. Identify proximity to bus terminals

## Part 4: Final Export and Analysis (Step 12)
12. Export complete dataset with analysis

**Total Runtime:** ~8-10 minutes for complete pipeline
- Part 1: ~2 minutes (without geocoding)
- Part 2: ~3 minutes
- Part 3: ~2-3 minutes
- Part 4: ~1 minute

---
# PART 1: H3 HEXAGON PROCESSING
---

## Step 1.1: Setup and Configuration

In [ ]:
# Install required packages (uncomment if needed)
# !pip install h3 geopandas pandas shapely geopy openpyxl

In [ ]:
import h3
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon
from shapely import wkt
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import warnings
import os
import sys
warnings.filterwarnings('ignore')

# Add src directory to path for config imports
project_root = os.path.abspath('.')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import configuration from src.config
from src.config import (
    H3_RESOLUTION,
    HUB_MERGE_THRESHOLD_M,
    MONTE_CARLO_ITERATIONS,
    MONTE_CARLO_RANDOM_SEED,
    MODE_WEIGHTS,
    TIER_NATIONAL,
    TIER_METRO,
    TIER_LOCAL,
    DISTANCE_DECAY_BETA,
    CRS_WGS84,
    CRS_ISRAEL_TM,
    ELIGIBILITY_MIN_PASSENGERS,
    ELIGIBILITY_MIN_MODES,
    REQUIRE_NON_RAIL_MODE,
    RAIL_ONLY_MODES,
    NON_RAIL_TRANSIT_MODES,
    AHP_ENABLED,
    AHP_EXPERT_CSV_PATH,
    MC_DIST_EXPORT_RAW_SCORES,
    MC_DIST_TOP_N_HUBS,
)

print("✓ All libraries and configuration loaded successfully!")
print(f"\nConfiguration from src/config.py:")
print(f"  H3_RESOLUTION: {H3_RESOLUTION}")
print(f"  HUB_MERGE_THRESHOLD_M: {HUB_MERGE_THRESHOLD_M}m")
print(f"  MONTE_CARLO_ITERATIONS: {MONTE_CARLO_ITERATIONS:,}")
print(f"  REQUIRE_NON_RAIL_MODE: {REQUIRE_NON_RAIL_MODE}")
print(f"  AHP_ENABLED: {AHP_ENABLED}")


## MASTER CONFIGURATION - ALL FILE PATHS

**Configure all input and output file paths here before running the pipeline.**

This single configuration section controls all file paths for the entire pipeline.

In [ ]:
# ============================================================================
# MASTER FILE PATH CONFIGURATION
# ============================================================================
# Configure ALL file paths here. Other cells will reference these variables.

import os

# ----------------------------------------------------------------------------
# INPUT FILES
# ----------------------------------------------------------------------------

# Part 1: Transit nodes and lines
INPUT_NODES_CSV = '/path/to/All_nodes+lines.csv'
LINES_MODE_CSV = '/path/to/Lines_and_Planned_Mode.csv'

# Part 2: Demand data and spatial layers
DEMAND_EXCEL = '/path/to/Nodes_w_results.xlsx'
MANUAL_DEMAND_UPDATES_CSV = '/path/to/manual_demand_updates.csv'  # Optional: CSV with node-level demand corrections
METRO_SHP = '/path/to/metro.shp'
DISTRICTS_SHP = '/path/to/districts.shp'

# Part 3: Demographic data
TAZ_SHAPEFILE = '/path/to/TAZ_2050.shp'
BUS_TERMINALS_SHP = '/path/to/bus_terminals.shp'  # Set to None if not available

# ----------------------------------------------------------------------------
# OUTPUT FILES
# ----------------------------------------------------------------------------

# Main output directory
OUTPUT_DIR = '/path/to/output'

# Part 1 outputs
OUTPUT_H3_HEXAGONS = os.path.join(OUTPUT_DIR, 'transit_h3_hexagons.csv')

# Part 2 outputs
OUTPUT_HUBS_WITH_DEMAND = os.path.join(OUTPUT_DIR, 'hubs_with_demand.csv')
OUTPUT_GROUPED_HUBS = os.path.join(OUTPUT_DIR, 'grouped_hubs.csv')

# Part 3 outputs
OUTPUT_FINAL = os.path.join(OUTPUT_DIR, 'hubs_complete.csv')
OUTPUT_FINAL_EXCEL = os.path.join(OUTPUT_DIR, 'hubs_complete.xlsx')

# Part 4 outputs
OUTPUT_SCORED_HUBS = os.path.join(OUTPUT_DIR, 'scored_hubs_final.csv')
OUTPUT_SCORED_EXCEL = os.path.join(OUTPUT_DIR, 'scored_hubs_final.xlsx')

# ----------------------------------------------------------------------------
# PROCESSING PARAMETERS (can be modified)
# ----------------------------------------------------------------------------

# Part 1: H3 and grouping
# H3_RESOLUTION imported from config              # H3 resolution (10 = ~15m hexagons)
BUFFER_DISTANCE = HUB_MERGE_THRESHOLD_M  # From config           # Buffer distance in meters for grouping
SKIP_GEOCODING = True           # Set to False to geocode addresses (slower)

# Part 2: Coordinate systems
CRS_PROJECTED = CRS_ISRAEL_TM  # From config    # Israel TM Grid (for meters)
# CRS_WGS84 imported from config        # WGS84 (for lat/lon)

# Part 3: Buffer zones for influence area analysis (in meters)
BUFFER_ZONES = {
    'zone1': (0, 600),      # 0-600m inner zone
    'zone2': (600, 1000),   # 600-1000m middle ring
    'zone3': (1000, 1200)   # 1000-1200m outer ring
}

# Part 4: Scoring parameters
# MONTE_CARLO_ITERATIONS imported from config  # Number of Monte Carlo simulation iterations
RANDOM_SEED = MONTE_CARLO_RANDOM_SEED  # From config                 # Random seed for reproducibility
DISTANCE_DECAY_BETA = 1.5        # Distance decay factor for population/employment scoring

# Hub tier names (Hebrew)
TIER_NATIONAL = "ארצי"           # National tier
TIER_METRO = "מטרופוליני"        # Metropolitan tier
TIER_LOCAL = "עירוני"            # Local tier

# Mode weights for service scoring (higher = more important mode)
# MODE_WEIGHTS imported from config
# Original definition (now in src/config.py):
# MODE_WEIGHTS = {
#     'Funicular': 1.0,
#     'Cable Line': 2.0,
#     'BRT': 3.0,
#     'LRT': 4.0,
#     'Metro': 5.0,
#     'Suburban Rail': 6.0,
#     'Interurban Rail': 7.0,
#     'HighSpeed Rail': 8.0,
#     'Rail': 7.0,
#     'Express Bus': 3.0,
#     'Bus': 2.0,
# }

## Step 1.2: Configure File Paths for Part 1

In [ ]:
# ============================================================================
# PART 1 CONFIGURATION - H3 HEXAGON PROCESSING
# ============================================================================
# File paths are configured in the MASTER CONFIGURATION section above.
# This section just displays what will be used.

print("Part 1 Configuration:")
print(f"  H3 Resolution: {H3_RESOLUTION}")
print(f"  Buffer Distance: {BUFFER_DISTANCE}m (edge-to-edge)")
print(f"  Skip Geocoding: {SKIP_GEOCODING}")
print(f"  Input Nodes: {INPUT_NODES_CSV}")
print(f"  Input Lines/Mode: {LINES_MODE_CSV}")
print(f"  Output: {OUTPUT_H3_HEXAGONS}")
print(f"  CRS Projected: {CRS_PROJECTED}")
print(f"  CRS WGS84: {CRS_WGS84}")


## Step 1.3: Load Transit Nodes Data

In [ ]:
print("Loading transit nodes data...")

# Read CSV with pandas
df = pd.read_csv(INPUT_NODES_CSV, encoding='windows-1255')

# Check if geometry column exists
if 'geometry' in df.columns:
    # If geometry exists as WKT string, convert it
    df['geometry'] = df['geometry'].apply(lambda x: wkt.loads(x) if pd.notna(x) else None)
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs=CRS_PROJECTED)
elif 'X' in df.columns and 'Y' in df.columns:
    # Create geometry from X, Y coordinates
    geometry = [Point(x, y) for x, y in zip(df['X'], df['Y'])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=CRS_PROJECTED)
else:
    raise ValueError("CSV must contain either 'geometry' column or both 'X' and 'Y' columns")

print(f"✓ Loaded {len(gdf)} records")
print(f"  CRS: {gdf.crs}")
print(f"  Columns: {list(gdf.columns)}")
print(f"\nFirst 3 rows:")
gdf.head(3)

## Step 1.4: Assign H3 Indices and Aggregate Lines

In [ ]:
print("Step 1.4: Loading Lines and Planned Mode data, then assigning H3 indices...")

print(f"  Loading mode data from: {LINES_MODE_CSV}")
lines_mode = pd.read_csv(LINES_MODE_CSV, encoding='windows-1255')
print(f"  ✓ Loaded {len(lines_mode)} line records")
print(f"  Columns: {list(lines_mode.columns)}")

# Convert to WGS84 for H3
gdf_wgs84 = gdf.to_crs(CRS_WGS84)

# Extract lat/lon
gdf_wgs84['lat'] = gdf_wgs84.geometry.y
gdf_wgs84['lon'] = gdf_wgs84.geometry.x

# Assign H3 index
gdf_wgs84['h3_index'] = gdf_wgs84.apply(
    lambda row: h3.latlng_to_cell(row['lat'], row['lon'], H3_RESOLUTION),
    axis=1
)

print(f"  ✓ Assigned H3 indices to {len(gdf_wgs84)} records")
print(f"  Unique H3 hexagons (before filtering): {gdf_wgs84['h3_index'].nunique()}")

# ============================================================================
# Filter out old Metronit lines (Haifa area, starting with 'm')
# ============================================================================
print("\n  Filtering out old Metronit lines...")
remove_old_metronit = []
if 'Area' in lines_mode.columns:
    haifa_lines = lines_mode[lines_mode['Area'] == 'Haifa']['Line_ModelName'].tolist()
    for line in haifa_lines:
        if str(line)[:1] == 'm':
            remove_old_metronit.append(line)
            
    if remove_old_metronit:
        print(f"    Removing old Metronit: {', '.join(map(str, remove_old_metronit))}")
        gdf_wgs84 = gdf_wgs84[~gdf_wgs84['LINE_ID'].isin(remove_old_metronit)]
        print(f"    ✓ After filtering: {len(gdf_wgs84)} records")
    else:
        print(f"    No old Metronit lines found")
else:
    print(f"    ⚠ 'Area' column not found in Lines_and_Planned_Mode.csv, skipping Metronit filter")

# ============================================================================
# Filter out Netanya LRT151/152 lines
# ============================================================================
print("\n  Filtering out Netanya LRT151/152 lines...")
remove_netanya15 = []
if 'Area' in lines_mode.columns:
    netanya_lines = lines_mode[lines_mode['Area'] == 'Netanya']['Line_ModelName'].tolist()
    for line in netanya_lines:
        if line == 'LRT151' or line == 'LRT152':
            remove_netanya15.append(line)
            
    if remove_netanya15:
        print(f"    Removing Netanya lines: {', '.join(map(str, remove_netanya15))}")
        gdf_wgs84 = gdf_wgs84[~gdf_wgs84['LINE_ID'].isin(remove_netanya15)]
        print(f"    ✓ After filtering: {len(gdf_wgs84)} records")
    else:
        print(f"    No Netanya LRT151/152 lines found")
else:
    print(f"    ⚠ 'Area' column not found, skipping Netanya filter")

# ============================================================================
# Merge with Lines and Planned Mode to get Mode_Planned
# ============================================================================
print("\n  Merging with mode data...")
gdf_with_mode = gdf_wgs84.merge(
    lines_mode, 
    left_on='LINE_ID', 
    right_on='Line_ModelName', 
    how='left',
    validate='many_to_many'
)

print(f"  ✓ Merged data: {len(gdf_with_mode)} records")

# Check if Mode_Planned column exists after merge
if 'Mode_Planned' not in gdf_with_mode.columns:
    print("  ⚠ WARNING: 'Mode_Planned' column not found after merge!")
    print(f"  Available columns: {list(gdf_with_mode.columns)}")
    print("  Using 'Bus' as default mode for all lines")
    gdf_with_mode['Mode_Planned'] = 'Bus'

# ============================================================================
# Group by h3_index, node, and Mode_Planned, then aggregate lines
# ============================================================================
print("\n  Aggregating by H3 hexagon and mode...")

h3_grouped = gdf_with_mode.groupby(['h3_index', 'node', 'Mode_Planned']).agg({
    'Line_ModelName': ['nunique', lambda x: list(x.unique())]
}).reset_index()

# Flatten column names
h3_grouped.columns = ['h3_index', 'node', 'Mode_Planned', 'Line_Nunique', 'Line_Unique']

print(f"  ✓ Created {len(h3_grouped)} unique H3-node-mode combinations")

# ============================================================================
# Further aggregate by h3_index only (combining all modes)
# ============================================================================
print("\n  Final aggregation by H3 hexagon...")

h3_final = h3_grouped.groupby('h3_index').agg({
    'node': 'first',
    'Mode_Planned': lambda x: list(x.unique()),  # List of all modes
    'Line_Nunique': 'sum',  # Total unique lines across all modes
    'Line_Unique': lambda x: list(set([item for sublist in x for item in sublist]))  # Flatten and deduplicate lines
}).reset_index()

print(f"  ✓ Aggregated to {len(h3_final)} unique hexagons")

# ============================================================================
# Create geometry from H3 indices
# ============================================================================
def h3_to_polygon(h3_index):
    """Convert H3 index to Shapely Polygon."""
    boundary = h3.cell_to_boundary(h3_index)
    # H3 returns (lat, lon), need (lon, lat) for Shapely
    return Polygon([(lon, lat) for lat, lon in boundary])

print("  Creating hexagon geometries...")
h3_final['geometry'] = h3_final['h3_index'].apply(h3_to_polygon)

# Convert to GeoDataFrame
gdf_h3 = gpd.GeoDataFrame(h3_final, geometry='geometry', crs=CRS_WGS84)

print(f"\n✓ Step 1.4 complete! Created {len(gdf_h3)} hexagons")
print(f"  Modes distribution:")
if 'Mode_Planned' in gdf_h3.columns:
    all_modes = [mode for modes_list in gdf_h3['Mode_Planned'] for mode in modes_list]
    mode_counts = pd.Series(all_modes).value_counts()
    for mode, count in mode_counts.items():
        print(f"    {mode}: {count}")

gdf_h3.head(3)

## Step 1.5: Create Groups Based on 120m Buffer (Edge-to-Edge)

In [ ]:
print(f"Step 1.5: Creating groups based on {BUFFER_DISTANCE}m buffer...")
print("Note: Measuring edge-to-edge distance (border to border)")
print("Note: Using transitive grouping (A near B, B near C → all same group)")

# Convert to projected CRS for meter-based operations
gdf_proj = gdf_h3.to_crs(CRS_PROJECTED).copy()
gdf_proj = gdf_proj.reset_index(drop=True)

n = len(gdf_proj)
print(f"  Processing {n} hexagons...")

In [ ]:
# Build spatial index for efficient querying
print("  Building spatial index...")
sindex = gdf_proj.sindex

# Union-Find data structure
parent = list(range(n))

def find(x):
    """Find root with path compression."""
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    """Union two components."""
    root_x = find(x)
    root_y = find(y)
    if root_x != root_y:
        parent[root_y] = root_x

print("  Union-Find structure initialized")

In [ ]:
# Find all pairs of hexagons within buffer_distance (edge to edge)
print("  Finding neighbors within buffer distance (edge-to-edge)...")
pairs_found = 0

# Add small tolerance for touching hexagons (floating point precision)
tolerance = 0.1  # 10cm tolerance for "touching"
effective_distance = BUFFER_DISTANCE + tolerance

for idx1 in range(n):
    geom1 = gdf_proj.iloc[idx1].geometry
    
    # Query spatial index for potential neighbors
    minx, miny, maxx, maxy = geom1.bounds
    search_bounds = (
        minx - BUFFER_DISTANCE,
        miny - BUFFER_DISTANCE,
        maxx + BUFFER_DISTANCE,
        maxy + BUFFER_DISTANCE
    )
    
    possible_neighbors = list(sindex.intersection(search_bounds))
    
    # Check actual edge-to-edge distance
    for idx2 in possible_neighbors:
        if idx2 <= idx1:  # Avoid checking pairs twice
            continue
        
        geom2 = gdf_proj.iloc[idx2].geometry
        distance = geom1.distance(geom2)
        
        if distance <= effective_distance:
            union(idx1, idx2)
            pairs_found += 1
    
    if (idx1 + 1) % 100 == 0:
        print(f"    Processed {idx1 + 1}/{n} hexagons...")

print(f"  ✓ Found {pairs_found} neighbor pairs within {BUFFER_DISTANCE}m (edge-to-edge)")

In [ ]:
# Assign group labels
print("  Assigning group IDs...")
root_to_group = {}
group_counter = 0

for idx in range(n):
    root = find(idx)
    if root not in root_to_group:
        root_to_group[root] = group_counter
        group_counter += 1

labels = [root_to_group[find(idx)] for idx in range(n)]
gdf_h3['group'] = labels
n_groups = len(root_to_group)

# Statistics
group_sizes = gdf_h3.groupby('group').size()

print(f"\n✓ Step 1.5 complete!")
print(f"  Created {n_groups} groups from {len(gdf_h3)} hexagons")
print(f"  Single hexagon groups: {(group_sizes == 1).sum()}")
print(f"  Multi-hexagon groups: {(group_sizes > 1).sum()}")
print(f"  Largest group: {group_sizes.max()} hexagons")
print(f"  Average group size: {group_sizes.mean():.2f} hexagons")

## Step 1.6: Geocode Addresses (Optional)

In [ ]:
if not SKIP_GEOCODING:
    print("Step 1.6: Geocoding addresses...")
    print("Note: This step uses Nominatim and may take several minutes.")
    print("      Rate limited to 1 request per second.\n")
    
    # Initialize geocoder
    geolocator = Nominatim(user_agent='transit_h3_processor')
    geocode = RateLimiter(geolocator.reverse, min_delay_seconds=1)
    
    # Get centroids in WGS84
    gdf_wgs84_final = gdf_h3.to_crs(CRS_WGS84)
    centroids = gdf_wgs84_final.geometry.centroid
    
    addresses = []
    total = len(centroids)
    
    for idx, centroid in enumerate(centroids):
        try:
            location = geocode(f"{centroid.y}, {centroid.x}", language='he')
            address = location.address if location else "Address not found"
            addresses.append(address)
            
            if (idx + 1) % 10 == 0:
                print(f"  Geocoded {idx + 1}/{total} addresses ({(idx+1)/total*100:.1f}%)")
        except Exception as e:
            print(f"  Error geocoding index {idx}: {e}")
            addresses.append("Geocoding error")
    
    gdf_h3['address'] = addresses
    print(f"\n✓ Step 1.6 complete! Geocoded {len(addresses)} locations")
    
else:
    print("Step 1.6: Skipping geocoding (as configured)")
    gdf_h3['address'] = 'Not geocoded'
    print("✓ Step 1.6 complete (skipped)")

## Step 1.7: Create Mode-Specific Line Count Columns

**PLACEHOLDER FIX** - This step creates mode-specific line count columns needed for scoring.

Currently uses a workaround that distributes lines evenly across modes based on `Mode_Planned`.

**Long-term solution:** Track lines per mode during initial aggregation in Part 1.

In [ ]:
# ============================================================================
# Step 1.7: Create Mode-Specific Line Count Columns
# ============================================================================
print("\n" + "="*80)
print("Step 1.7: Creating mode-specific line count columns...")
print("="*80)

# Define mode-specific line columns
MODE_LINE_COLS = [
    'BRT Lines', 'Cable Line Lines', 'Funicular Lines',
    'HighSpeed Rail Lines', 'Interurban Rail Lines', 'LRT Lines',
    'Metro Lines', 'Suburban Rail Lines'
]

# Initialize all mode-specific line columns to 0
for col in MODE_LINE_COLS:
    gdf_h3[col] = 0

# WORKAROUND: Map modes to column names (excluding Bus)
MODE_TO_COLUMN = {
    'BRT': 'BRT Lines',
    'Metro': 'Metro Lines',
    'LRT': 'LRT Lines',
    'Light Rail': 'LRT Lines',
    'Rail': 'Interurban Rail Lines',
    'Interurban Rail': 'Interurban Rail Lines',
    'HighSpeed Rail': 'HighSpeed Rail Lines',
    'Suburban Rail': 'Suburban Rail Lines',
    'Cable Line': 'Cable Line Lines',
    'Funicular': 'Funicular Lines',
    'Bus': None,  # Exclude bus from mode line counts
    'Express Bus': None,
}

def create_mode_line_columns(row):
    """Create mode-specific line count columns by distributing lines across modes.

    WORKAROUND: Distributes Line_Nunique evenly across valid modes in Mode_Planned.
    This is approximate but allows scoring to work.
    """
    mode_counts = {}
    for col in MODE_LINE_COLS:
        mode_counts[col] = 0

    # Get modes for this row
    modes = row['Mode_Planned']
    
    # Handle None, NaN, or empty list cases
    if modes is None or (isinstance(modes, list) and len(modes) == 0):
        return pd.Series(mode_counts)
    
    # Parse modes - Mode_Planned is already a list from aggregation
    if isinstance(modes, str):
        # Handle legacy case where it might be a string
        mode_list = [m.strip() for m in modes.split(',')]
    elif isinstance(modes, list):
        # Mode_Planned is typically a list
        mode_list = modes
    else:
        # Fallback for other cases
        mode_list = []

    # Filter to valid modes (excluding Bus)
    valid_modes = [m for m in mode_list if MODE_TO_COLUMN.get(m) is not None]

    if not valid_modes:
        return pd.Series(mode_counts)

    # Distribute Line_Nunique evenly across valid modes
    total_lines = row.get('Line_Nunique', 0)
    lines_per_mode = total_lines / len(valid_modes) if total_lines > 0 else 0

    for mode in valid_modes:
        col = MODE_TO_COLUMN.get(mode)
        if col:
            mode_counts[col] = lines_per_mode

    return pd.Series(mode_counts)

# Apply to create columns
print("  Distributing lines across modes...")
mode_line_df = gdf_h3.apply(create_mode_line_columns, axis=1)

for col in MODE_LINE_COLS:
    gdf_h3[col] = mode_line_df[col]

print(f"✓ Created {len(MODE_LINE_COLS)} mode-specific line count columns")
print(f"\n  Mode distribution:")
for col in MODE_LINE_COLS:
    nonzero = (gdf_h3[col] > 0).sum()
    if nonzero > 0:
        total_lines = gdf_h3[col].sum()
        print(f"    {col}: {nonzero} hubs, {total_lines:.0f} total lines")

print(f"\n✓ Step 1.7 complete!")

## Step 1.8: Export H3 Hexagons with Groups

In [ ]:
print(f"Step 1.8: Exporting H3 hexagons to {OUTPUT_H3_HEXAGONS}...")

# Select final columns
output_columns = [
    'h3_index', 'node', 'Mode_Planned', 'Line_Nunique', 
    'Line_Unique', 'geometry', 'group', 'address',
    # Mode-specific line count columns (created in Step 1.7)
    'BRT Lines', 'Cable Line Lines', 'Funicular Lines',
    'HighSpeed Rail Lines', 'Interurban Rail Lines', 'LRT Lines',
    'Metro Lines', 'Suburban Rail Lines'
]

# Only include columns that actually exist in gdf_h3
output_columns = [col for col in output_columns if col in gdf_h3.columns]

gdf_output_part1 = gdf_h3[output_columns].copy()

# CRITICAL: Clean node column to ensure it's a single integer value
# This prevents '[np.int64(123)]' format in CSV that breaks Part 2
print("  Cleaning node column...")
if 'node' in gdf_output_part1.columns:
    # If node is somehow a list, take the first value
    gdf_output_part1['node'] = gdf_output_part1['node'].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x
    )
    # Convert numpy types to Python int
    gdf_output_part1['node'] = gdf_output_part1['node'].apply(
        lambda x: int(x) if pd.notna(x) else x
    )
    print(f"  ✓ Node column cleaned: {gdf_output_part1['node'].dtype}")

# Ensure output directory exists
os.makedirs(os.path.dirname(OUTPUT_H3_HEXAGONS) if os.path.dirname(OUTPUT_H3_HEXAGONS) else '.', exist_ok=True)

# Export
if OUTPUT_H3_HEXAGONS.endswith('.csv'):
    gdf_output_part1['geometry'] = gdf_output_part1['geometry'].apply(lambda x: x.wkt)
    gdf_output_part1.to_csv(OUTPUT_H3_HEXAGONS, index=False, encoding='utf-8-sig')
elif OUTPUT_H3_HEXAGONS.endswith('.geojson'):
    gdf_output_part1.to_file(OUTPUT_H3_HEXAGONS, driver='GeoJSON')
elif OUTPUT_H3_HEXAGONS.endswith('.gpkg'):
    gdf_output_part1.to_file(OUTPUT_H3_HEXAGONS, driver='GPKG')
else:
    gdf_output_part1.to_file(OUTPUT_H3_HEXAGONS + '.geojson', driver='GeoJSON')

print(f"\n✓ Step 1.8 complete! File saved successfully.")
print(f"\n" + "="*80)
print("PART 1 COMPLETE - H3 HEXAGON PROCESSING")
print("="*80)
print(f"\nOutput: {OUTPUT_H3_HEXAGONS}")
print(f"Records: {len(gdf_output_part1)} INDIVIDUAL hexagons")
print(f"Groups: {gdf_output_part1['group'].nunique()} proximity groups identified")
print(f"\nColumns: {list(gdf_output_part1.columns)}")
print(f"\nNote: Grouping will happen in Part 2 AFTER demand assignment")
print(f"Ready for Part 2: Demand Data Processing")


---
# PART 2: DEMAND DATA PROCESSING
---

This part adds daily demand (boardings, alightings, transfers) to each hub group.

**What it does:**
1. Tags hubs with geographic area (from metro/districts shapefiles)
2. Loads demand data from Excel file with regional model sheets
3. Matches demand to nodes based on area (each region uses a different model)
4. Handles overlay models (Hadera, Haifa Metronit) that add to existing demand

**Regional Models:**
- Haifa (sheet: 5040_Daily)
- Tel Aviv (sheet: Daily_5087)
- Beer Sheva (sheet: Daily_BS)
- Jerusalem (sheet: Daily_Jerusalem)
- Ashdod-Ashkelon (sheet: Daily_5093)
- Hadera (overlay model)
- Haifa Metronit (overlay model)
- Rail (national)

**Required Files:**
- Metro shapefile with METRO_NAME column
- Districts shapefile with MACHOZ column
- Demand Excel file with regional sheets

## Step 2.1: Configure Paths for Part 2

In [ ]:
# ============================================================================
# PART 2 CONFIGURATION - DEMAND DATA PROCESSING
# ============================================================================
# File paths are configured in the MASTER CONFIGURATION section above.
# This section just displays what will be used and defines data mappings.

# Input: H3 hexagons from Part 1 (automatically set)
INPUT_H3_FOR_DEMAND = OUTPUT_H3_HEXAGONS

# Sheet name mapping: Excel sheet names -> region names
SHEET_NAME_MAPPING = {
    '5040_Daily': 'Haifa',
    'Daily_5087': 'Tel Aviv',
    'Daily_BS': 'Beer Sheva',
    'Daily_Hadera': 'Hadera',
    'Daily_Jerusalem': 'Jerusalem',
    'HaifaNewMetronit': 'Haifa Metronit',
    'Daily_5093': 'Ashdod-Ashkelon',
    'National': 'Rail',
    # Also accept already-mapped names
    'Haifa': 'Haifa',
    'TelAviv': 'Tel Aviv',
    'Tel Aviv': 'Tel Aviv',
    'BeerSheva': 'Beer Sheva',
    'Beer Sheva': 'Beer Sheva',
    'Hadera': 'Hadera',
    'Jerusalem': 'Jerusalem',
    'Ashdod': 'Ashdod-Ashkelon',
    'Ashdod-Ashkelon': 'Ashdod-Ashkelon',
    'Ashkelon': 'Ashdod-Ashkelon',
    'HaifaMetronit': 'Haifa Metronit',
    'Haifa Metronit': 'Haifa Metronit',
}

# Area to region mapping (Hebrew area names from spatial layers -> demand model regions)
AREA_TO_REGION = {
    'חיפה': 'Haifa',
    'צפון': 'Haifa',
    'תל אביב': 'Tel Aviv',
    'תל-אביב': 'Tel Aviv',
    'מרכז': 'Tel Aviv',
    'באר שבע': 'Beer Sheva',
    'דרום': 'Beer Sheva',
    'ירושלים': 'Jerusalem',
    'אשדוד': 'Ashdod-Ashkelon',
    'אשקלון': 'Ashdod-Ashkelon',
    # English names
    'Haifa': 'Haifa',
    'North': 'Haifa',
    'Tel Aviv': 'Tel Aviv',
    'Center': 'Tel Aviv',
    'Beer Sheva': 'Beer Sheva',
    'South': 'Beer Sheva',
    'Jerusalem': 'Jerusalem',
}

print("Part 2 Configuration:")
print(f"  Input H3: {INPUT_H3_FOR_DEMAND}")
print(f"  Demand Excel: {DEMAND_EXCEL}")
print(f"  Metro Shapefile: {METRO_SHP}")
print(f"  Districts Shapefile: {DISTRICTS_SHP}")
print(f"  Output (ungrouped): {OUTPUT_HUBS_WITH_DEMAND}")
print(f"  Output (grouped): {OUTPUT_GROUPED_HUBS}")
print(f"\nSheet name mappings: {len(SHEET_NAME_MAPPING)} entries")
print(f"Area to region mappings: {len(AREA_TO_REGION)} entries")


<cell_type>markdown</cell_type>## Step 2.2: Load H3 Hexagons from Part 1

In [ ]:
print("Step 2.2: Loading H3 hexagons from Part 1...")

# Check if file exists
if not os.path.exists(INPUT_H3_FOR_DEMAND):
    print(f"✗ Input file not found: {INPUT_H3_FOR_DEMAND}")
    print("  Make sure Part 1 completed successfully and the path is correct.")
    PART2_AVAILABLE = False
else:
    # Load H3 hexagons
    df_h3 = pd.read_csv(INPUT_H3_FOR_DEMAND, encoding='utf-8-sig')
    
    # Parse geometry from WKT
    if 'geometry' in df_h3.columns and df_h3['geometry'].dtype == 'object':
        df_h3['geometry'] = df_h3['geometry'].apply(lambda x: wkt.loads(x) if pd.notna(x) else None)
    
    gdf_demand = gpd.GeoDataFrame(df_h3, geometry='geometry', crs=CRS_WGS84)
    
    print(f"✓ Loaded {len(gdf_demand)} H3 hexagons")
    print(f"  Columns: {list(gdf_demand.columns)}")
    print(f"  Groups: {gdf_demand['group'].nunique()}")
    
    # Initialize demand columns
    gdf_demand['TotalDemand'] = 0.0
    gdf_demand['TotalTransfers'] = 0.0
    gdf_demand['area'] = None
    
    PART2_AVAILABLE = True
    
    gdf_demand.head(3)

## Step 2.3: Tag Hubs with Area and Location from Spatial Layers

This step assigns each hub to a geographic area and metropolitan location using spatial joins:
1. First try metro shapefile:
   - METRO_NAME column → 'area' (e.g., 'תל אביב', 'חיפה')
   - ZONE_NAME column → 'location' (e.g., 'גלעין', 'טבעת', 'פריפריה')
2. Fall back to districts shapefile (MACHOZ column) for both area and location

The 'area' determines which regional demand model to use for matching.
The 'location' determines metropolitan position scoring (Core/Ring/Periphery).

In [ ]:
if PART2_AVAILABLE:
    print("Step 2.3: Tagging hubs with area and location from spatial layers...")

    # ============================================================================
    # Helper function: Try multiple encodings for shapefile reading
    # FIXED: Prioritize windows-1255 for Hebrew data
    # ============================================================================
    def is_valid_hebrew_text(text):
        """
        Check if text contains valid Hebrew characters.
        Returns True if text looks like valid Hebrew, False if gibberish.
        """
        if not isinstance(text, str) or len(text) == 0:
            return True  # Empty/None is ok
        
        # Hebrew Unicode range: 0x0590-0x05FF
        hebrew_chars = sum(1 for c in text if '\u0590' <= c <= '\u05FF')
        # Check if at least 30% of non-space chars are Hebrew
        non_space = sum(1 for c in text if not c.isspace())
        if non_space == 0:
            return True
        
        hebrew_ratio = hebrew_chars / non_space
        return hebrew_ratio > 0.3
    
    def validate_hebrew_in_gdf(gdf, columns_to_check):
        """
        Check if a GeoDataFrame has valid Hebrew text in specified columns.
        Returns True if Hebrew looks valid, False if gibberish.
        """
        for col in columns_to_check:
            if col not in gdf.columns:
                continue
            # Check first few non-null values
            sample_values = gdf[col].dropna().head(5).tolist()
            for val in sample_values:
                if not is_valid_hebrew_text(str(val)):
                    return False
        return True
    
    def read_shapefile_with_encoding(filepath, name="shapefile", hebrew_columns=None):
        """
        Read shapefile with encoding strategies optimized for Hebrew data.
        
        FIXED: Now prioritizes windows-1255 (Hebrew) encoding and validates
        that Hebrew text is readable, not gibberish.
        
        Args:
            filepath: Path to shapefile
            name: Name for error messages
            hebrew_columns: List of column names expected to contain Hebrew
        
        Returns tuple: (GeoDataFrame, encoding_used)
        """
        # FIXED: Prioritize Hebrew encodings first
        encodings_to_try = [
            ('windows-1255', "Windows-1255 (Hebrew)"),
            ('cp1255', "CP1255 (Hebrew)"),
            ('ISO-8859-8', "ISO-8859-8 (Hebrew)"),
            ('utf-8', "UTF-8"),
            (None, "auto-detect"),
        ]
        
        errors = []
        
        for encoding, encoding_name in encodings_to_try:
            try:
                if encoding is None:
                    gdf = gpd.read_file(filepath)
                else:
                    gdf = gpd.read_file(filepath, encoding=encoding)
                
                # FIXED: Validate Hebrew text is readable
                if hebrew_columns:
                    if validate_hebrew_in_gdf(gdf, hebrew_columns):
                        print(f"    ✓ Successfully loaded with encoding: {encoding_name}")
                        return gdf, encoding_name
                    else:
                        errors.append(f"      {encoding_name}: Hebrew text is gibberish")
                        continue
                else:
                    print(f"    ✓ Successfully loaded with encoding: {encoding_name}")
                    return gdf, encoding_name
                    
            except Exception as e:
                errors.append(f"      {encoding_name}: {str(e)[:100]}")
                continue
        
        # If all encodings failed, print errors and raise
        print(f"    ✗ Failed to load {name} with any encoding:")
        for error in errors:
            print(error)
        raise RuntimeError(f"Could not load {filepath} with any encoding strategy")

    def fix_truncated_hebrew(text):
        """
        Fix truncated Hebrew text by restoring the missing final letter.

        Common truncations:
        - 'גלעי' -> 'גלעין' (Core)
        - 'טבעת פנימי' -> 'טבעת פנימית' (Inner Ring)
        - 'טבעת חיצוני' -> 'טבעת חיצונית' (Outer Ring)
        - 'טבעת תיכונ' -> 'טבעת תיכונה' (Middle Ring)
        - Any word ending with 'י' that should end with 'ית' (feminine adjective)

        Args:
            text: Hebrew text string that may be truncated

        Returns:
            Fixed Hebrew text with proper final letters
        """
        if not isinstance(text, str) or not text:
            return text

        # Known truncation patterns and their fixes
        # Format: (truncated_pattern, correct_replacement)
        fixes = [
            # Exact matches (most specific)
            (r'^גלעי$', 'גלעין'),  # Core (exact word)
            (r'^טבעת פנימי$', 'טבעת פנימית'),  # Inner Ring
            (r'^טבעת חיצוני$', 'טבעת חיצונית'),  # Outer Ring
            (r'^טבעת תיכונ$', 'טבעת תיכונה'),  # Middle Ring
            (r'^טבע$', 'טבעת'),  # Ring (if severely truncated)

            # Word-level fixes (match within larger strings)
            (r'\bגלעי\b', 'גלעין'),  # Core (as word)
            (r'\bטבעת פנימי\b', 'טבעת פנימית'),  # Inner Ring (as phrase)
            (r'\bטבעת חיצוני\b', 'טבעת חיצונית'),  # Outer Ring (as phrase)
            (r'\bטבעת תיכונ\b', 'טבעת תיכונה'),  # Middle Ring (as phrase)

            # General pattern: Hebrew word ending with 'י' should end with 'ית' (feminine)
            # Only apply if the word is likely an adjective (e.g., after 'טבעת')
            (r'(טבעת\s+\S*?)י(?=\s|$)', r'\1ית'),  # Any adjective after טבעת ending in י
        ]

        fixed_text = text.strip()

        # Apply each fix pattern
        for pattern, replacement in fixes:
            fixed_text = re.sub(pattern, replacement, fixed_text)

        # If text changed, log it
        if fixed_text != text.strip():
            # This will be visible during notebook execution
            pass  # Logging will be done by caller

        return fixed_text

    def fix_truncated_hebrew_in_gdf(gdf, columns):
        """
        Apply Hebrew text fixes to specified columns in a GeoDataFrame.

        Args:
            gdf: GeoDataFrame to fix
            columns: List of column names to fix

        Returns:
            Number of values fixed
        """
        fixes_count = 0

        for col in columns:
            if col not in gdf.columns:
                continue

            # Apply fix to non-null string values
            original_values = gdf[col].copy()
            gdf[col] = gdf[col].apply(
                lambda x: fix_truncated_hebrew(x) if pd.notna(x) and isinstance(x, str) else x
            )

            # Count how many were fixed
            changed = (original_values != gdf[col]) & original_values.notna()
            if changed.any():
                fixes_count += changed.sum()
                print(f"    ✓ Fixed {changed.sum()} truncated values in column '{col}'")
                # Show examples of fixes
                for idx in gdf[changed].index[:3]:  # Show first 3 examples
                    old_val = original_values.loc[idx]
                    new_val = gdf.loc[idx, col]
                    print(f"      '{old_val}' -> '{new_val}'")

        return fixes_count


    def check_hebrew_truncation(gdf, columns_to_check, encoding_name):
        """
        Check if Hebrew text appears truncated by looking for common patterns.
        Returns True if truncation is suspected.
        """
        truncation_indicators = [
            'גלעי',  # Should be 'גלעין'
            'תל אבי',  # Should be 'תל אביב'
            'חיפ',  # Should be 'חיפה'
            'ירושלי',  # Should be 'ירושלים'
        ]
        
        for col in columns_to_check:
            if col not in gdf.columns:
                continue
            values = gdf[col].dropna().astype(str).unique()
            for value in values:
                for indicator in truncation_indicators:
                    if indicator in value:
                        print(f"    ⚠ WARNING: Possible truncation detected in column '{col}'")
                        print(f"      Value '{value}' may be truncated (encoding: {encoding_name})")
                        print(f"      Common pattern '{indicator}' found")
                        return True
        return False

    def diagnose_shapefile_content(gdf, name, encoding_name, key_columns):
        """Print diagnostic information about loaded shapefile."""
        print(f"\n    Diagnostic info for {name} (encoding: {encoding_name}):")
        print(f"      Rows: {len(gdf)}")
        print(f"      Columns: {list(gdf.columns)}")
        for col in key_columns:
            if col in gdf.columns:
                unique_vals = gdf[col].dropna().unique().tolist()
                print(f"      {col} values ({len(unique_vals)}): {unique_vals[:10]}")  # Show first 10
                # Check string lengths
                if len(unique_vals) > 0:
                    str_vals = [str(v) for v in unique_vals]
                    avg_len = sum(len(s) for s in str_vals) / len(str_vals)
                    print(f"        Average length: {avg_len:.1f} characters")
                    # Check for very short values (possible truncation)
                    short_vals = [v for v in str_vals if len(v) < 4]
                    if short_vals:
                        print(f"        ⚠ Short values (< 4 chars): {short_vals}")

    metro_gdf = None
    districts_gdf = None

    # Initialize columns
    gdf_demand['location'] = None

    # ============================================================================
    # Load metro shapefile with encoding detection
    # ============================================================================
    if os.path.exists(METRO_SHP):
        print(f"\n  Loading metro shapefile: {METRO_SHP}")
        try:
            # FIXED: Pass expected Hebrew columns for validation
            metro_hebrew_cols = ['METRO_NAME', 'ZONE_NAME', 'MetroName', 'ZoneName', 'NAME', 'SHEM']
            metro_gdf, metro_encoding = read_shapefile_with_encoding(
                METRO_SHP, "metro", hebrew_columns=metro_hebrew_cols
            )
            print(f"    ✓ Loaded {len(metro_gdf)} metro features")

            # Diagnose content
            diagnose_shapefile_content(
                metro_gdf,
                "metro layer",
                metro_encoding,
                ['METRO_NAME', 'ZONE_NAME', 'MetroName', 'ZoneName']
            )

            # Check for truncation
            check_hebrew_truncation(
                metro_gdf,
                ['METRO_NAME', 'ZONE_NAME', 'MetroName', 'ZoneName'],
                metro_encoding
            )
        except Exception as e:
            print(f"    ✗ Failed to load metro shapefile: {e}")
            metro_gdf = None
    else:
        print(f"  ⚠ Metro shapefile not found: {METRO_SHP}")

    # ============================================================================
    # Load districts shapefile with encoding detection
    # ============================================================================
    if os.path.exists(DISTRICTS_SHP):
        print(f"\n  Loading districts shapefile: {DISTRICTS_SHP}")
        try:
            # FIXED: Pass expected Hebrew columns for validation
            districts_hebrew_cols = ['MACHOZ', 'SHEM_MACHOZ', 'SHEM_NAFA', 'District', 'NAME', 'SHEM']
            districts_gdf, districts_encoding = read_shapefile_with_encoding(
                DISTRICTS_SHP, "districts", hebrew_columns=districts_hebrew_cols
            )
            print(f"    ✓ Loaded {len(districts_gdf)} district features")

            # Diagnose content
            diagnose_shapefile_content(
                districts_gdf,
                "districts layer",
                districts_encoding,
                ['MACHOZ', 'SHEM_MACHOZ', 'SHEM_NAFA', 'District', 'NAME']
            )

            # Check for truncation
            check_hebrew_truncation(
                districts_gdf,
                ['MACHOZ', 'SHEM_MACHOZ', 'SHEM_NAFA', 'District', 'NAME'],
                districts_encoding
            )
        except Exception as e:
            print(f"    ✗ Failed to load districts shapefile: {e}")
            districts_gdf = None
    else:
        print(f"  ⚠ Districts shapefile not found: {DISTRICTS_SHP}")

    # ============================================================================
    # Tag with metro layer first (both area and location)
    # ============================================================================
    if metro_gdf is not None:
        try:
            print("\n  Tagging with metro layer...")
            hubs_wgs = gdf_demand.to_crs('EPSG:4326')
            metro_wgs = metro_gdf.to_crs('EPSG:4326')

            # Find metro name column (for 'area')
            metro_col = None
            for col in ['METRO_NAME', 'MetroName', 'metro_name', 'NAME', 'name', 'SHEM']:
                if col in metro_wgs.columns:
                    metro_col = col
                    break

            # Find zone name column (for 'location')
            zone_col = None
            for col in ['ZONE_NAME', 'ZoneName', 'zone_name', 'ZONE', 'zone']:
                if col in metro_wgs.columns:
                    zone_col = col
                    break

            if metro_col:
                print(f"    Using area column: {metro_col}")
                print(f"    Metro values: {metro_wgs[metro_col].unique().tolist()}")

                # Select columns for join
                join_cols = [metro_col, 'geometry']
                if zone_col:
                    print(f"    Using location column: {zone_col}")
                    print(f"    Zone values: {metro_wgs[zone_col].unique().tolist()}")
                    join_cols.insert(1, zone_col)
                else:
                    print(f"    ⚠ No zone column found - location will not be set from metro layer")

                # Spatial join
                joined = gpd.sjoin(hubs_wgs, metro_wgs[join_cols],
                                  how='left', predicate='intersects')

                if joined.index.duplicated().any():
                    joined = joined[~joined.index.duplicated(keep='first')]

                # Assign area
                gdf_demand['area'] = joined[metro_col].values
                n_tagged_area = gdf_demand['area'].notna().sum()
                print(f"    ✓ Tagged area for {n_tagged_area}/{len(gdf_demand)} hubs")

                # Assign location (if zone column exists)
                if zone_col:
                    gdf_demand['location'] = joined[zone_col].values
                    # Convert to list format for compatibility with scoring code
                    gdf_demand['location'] = gdf_demand['location'].apply(
                        lambda x: [x] if pd.notna(x) else None
                    )
                    n_tagged_loc = gdf_demand['location'].notna().sum()
                    print(f"    ✓ Tagged location for {n_tagged_loc}/{len(gdf_demand)} hubs")
            else:
                print(f"    ⚠ No metro name column found")
        except Exception as e:
            print(f"    ✗ Metro tagging failed: {e}")
            import traceback
            traceback.print_exc()

    # ============================================================================
    # Fall back to districts for untagged hubs
    # ============================================================================
    if districts_gdf is not None:
        try:
            nan_mask = gdf_demand['area'].isna()
            if nan_mask.any():
                print(f"\n  Tagging {nan_mask.sum()} untagged hubs with districts layer...")
                hubs_wgs = gdf_demand[nan_mask].to_crs('EPSG:4326')
                districts_wgs = districts_gdf.to_crs('EPSG:4326')

                # Find district name column
                district_col = None
                for col in ['MACHOZ', 'SHEM_MACHOZ', 'SHEM_NAFA', 'District', 'NAME', 'SHEM']:
                    if col in districts_wgs.columns:
                        district_col = col
                        break

                if district_col:
                    print(f"    Using column: {district_col}")
                    print(f"    District values: {districts_wgs[district_col].unique().tolist()}")

                    joined = gpd.sjoin(hubs_wgs, districts_wgs[[district_col, 'geometry']],
                                      how='left', predicate='within')

                    if joined.index.duplicated().any():
                        joined = joined[~joined.index.duplicated(keep='first')]

                    # Update only NaN rows (both area and location)
                    for idx in joined.index:
                        if pd.notna(joined.loc[idx, district_col]):
                            district_val = joined.loc[idx, district_col]
                            gdf_demand.loc[idx, 'area'] = district_val
                            # For districts, use district name as location (periphery)
                            if pd.isna(gdf_demand.loc[idx, 'location']) or gdf_demand.loc[idx, 'location'] is None:
                                gdf_demand.loc[idx, 'location'] = [district_val]

                    n_tagged = gdf_demand['area'].notna().sum()
                    print(f"    ✓ Total tagged: {n_tagged}/{len(gdf_demand)} hubs")
                else:
                    print(f"    ⚠ No district column found")
        except Exception as e:
            print(f"    ✗ District tagging failed: {e}")
            import traceback
            traceback.print_exc()

    # Fill remaining with 'Unknown'
    gdf_demand['area'] = gdf_demand['area'].fillna('Unknown')
    gdf_demand['location'] = gdf_demand['location'].apply(
        lambda x: ['Unknown'] if x is None or (isinstance(x, float) and pd.isna(x)) else x
    )

    # ============================================================================
    # Summary
    # ============================================================================
    print("\n  Area distribution:")
    area_counts = gdf_demand['area'].value_counts()
    for area, count in area_counts.items():
        print(f"    {area}: {count}")

    print("\n  Location distribution:")
    location_counts = gdf_demand['location'].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else str(x)
    ).value_counts()
    for loc, count in location_counts.items():
        print(f"    {loc}: {count}")


    # ============================================================================
    # FIX: Repair truncated Hebrew text in area and location columns
    # ============================================================================
    print("\n  Fixing any truncated Hebrew text...")

    # Fix area column (string values)
    if 'area' in gdf_demand.columns:
        original_areas = gdf_demand['area'].copy()
        gdf_demand['area'] = gdf_demand['area'].apply(
            lambda x: fix_truncated_hebrew(x) if pd.notna(x) else x
        )
        area_fixed = ((original_areas != gdf_demand['area']) & original_areas.notna()).sum()
        if area_fixed > 0:
            print(f"    ✓ Fixed {area_fixed} truncated values in 'area' column")
            # Show examples
            for idx in gdf_demand[(original_areas != gdf_demand['area']) & original_areas.notna()].index[:3]:
                print(f"      '{original_areas.loc[idx]}' -> '{gdf_demand.loc[idx, 'area']}'")

    # Fix location column (list values - fix each element)
    if 'location' in gdf_demand.columns:
        location_fixed = 0
        for idx in gdf_demand.index:
            loc_val = gdf_demand.loc[idx, 'location']
            if isinstance(loc_val, list):
                fixed_loc = [fix_truncated_hebrew(item) if isinstance(item, str) else item
                            for item in loc_val]
                if fixed_loc != loc_val:
                    location_fixed += 1
                    if location_fixed <= 3:  # Show first 3 examples
                        print(f"      '{loc_val}' -> '{fixed_loc}'")
                gdf_demand.at[idx, 'location'] = fixed_loc

        if location_fixed > 0:
            print(f"    ✓ Fixed {location_fixed} truncated values in 'location' column")

    print("  ✓ Hebrew text fix complete")

    print(f"\n✓ Step 2.3 complete!")

    # Diagnostic: Show sample values
    print("\n  Sample data verification:")
    if 'area' in gdf_demand.columns:
        print(f"    area column type: {gdf_demand['area'].dtype}")
        sample_areas = gdf_demand['area'].dropna().unique()[:3]
        print(f"    area samples: {list(sample_areas)}")
    if 'location' in gdf_demand.columns:
        print(f"    location column exists: True")
        sample_locs = gdf_demand[gdf_demand['location'].notna()]['location'].apply(
            lambda x: x[0] if isinstance(x, list) and len(x) > 0 else str(x)
        ).unique()[:3]
        print(f"    location samples: {list(sample_locs)}")

    # DIAGNOSTIC: Check area/location before aggregation
    print("\n  DIAGNOSTIC - After Step 2.3:")
    print(f"    gdf_demand shape: {gdf_demand.shape}")
    print(f"    'area' column exists: {'area' in gdf_demand.columns}")
    if 'area' in gdf_demand.columns:
        print(f"    'area' non-null count: {gdf_demand['area'].notna().sum()}/{len(gdf_demand)}")
        print(f"    'area' unique values (first 5): {gdf_demand['area'].dropna().unique()[:5].tolist()}")
    print(f"    'location' column exists: {'location' in gdf_demand.columns}")
    if 'location' in gdf_demand.columns:
        print(f"    'location' non-null count: {gdf_demand['location'].notna().sum()}/{len(gdf_demand)}")
        sample_locs = gdf_demand[gdf_demand['location'].notna()]['location'].head(3).tolist()
        print(f"    'location' sample values: {sample_locs}")
else:
    print("Step 2.3: Skipped (Part 2 not available)")


## Step 2.4: Per-Sheet Column Configuration

Different regional models use different column names for node IDs, boardings, and alightings.
This configuration maps each region to its specific column names.

In [ ]:
# Per-sheet column configurations
# Each entry maps a region to its column naming conventions
SHEET_COLUMN_CONFIG = {
    'Beer Sheva': {
        'node_cols': ['NODE_ID', 'Node', 'node'],
        'boardings_cols': ['Boardings', 'Boardings_Daily', 'TotalBoardings'],
        'alightings_cols': ['Alightings', 'Alightings_Daily', 'TotalAlight'],
        'transfer_cols': [],
    },
    'Hadera': {
        'node_cols': ['NodeID', 'Node', 'node'],
        'boardings_cols': ['On', 'Boardings', 'Boardings_Daily'],
        'alightings_cols': ['Off', 'Alightings', 'Alightings_Daily'],
        'transfer_cols': [],
    },
    'Tel Aviv': {
        'node_cols': ['Node', 'node'],
        'boardings_cols': ['TotalBoardings', 'Boardings', 'Boardings_Daily'],
        'alightings_cols': ['TotalAlight', 'Alightings', 'Alightings_Daily'],
        'transfer_cols': ['TransferBoardings', 'TransferAlight'],
    },
    'Ashdod-Ashkelon': {
        'node_cols': ['Node', 'node'],
        'boardings_cols': ['InitialBoardings', 'Boardings', 'TotalBoardings'],
        'alightings_cols': ['FinalAlight', 'Alightings', 'TotalAlight'],
        'transfer_cols': [],
    },
    'Haifa': {
        'node_cols': ['Node', 'node'],
        'boardings_cols': ['TotalBoardings', 'Boardings', 'Boardings_Daily'],
        'alightings_cols': ['TotalAlight', 'Alightings', 'Alightings_Daily'],
        'transfer_cols': ['TransferBoardings', 'TransferAlight'],
    },
    'Haifa Metronit': {
        'node_cols': ['ModelNode', 'Node', 'node'],
        'boardings_cols': ['Boardings_Daily', 'Boardings', 'TotalBoardings'],
        'alightings_cols': ['Alightings_Daily', 'Alightings', 'TotalAlight'],
        'transfer_cols': [],
    },
    'Jerusalem': {
        'node_cols': ['ID', 'Node', 'node'],
        'boardings_cols': ['DailyBoard_2050', 'Boardings', 'Boardings_Daily'],
        'alightings_cols': ['DailyAlight_2050', 'Alightings', 'Alightings_Daily'],
        'transfer_cols': [],
    },
    'Rail': {
        'node_cols': ['Node', 'node', 'NODE_ID'],
        'boardings_cols': ['Boardings', 'TotalBoardings', 'Boardings_Daily'],
        'alightings_cols': ['Alightings', 'TotalAlight', 'Alightings_Daily'],
        'transfer_cols': [],
    },
}

# Default column config for unknown sheets
DEFAULT_COLUMN_CONFIG = {
    'node_cols': ['Node', 'node', 'NODE_ID', 'NodeID', 'ID', 'ModelNode', 'N'],
    'boardings_cols': ['Boardings', 'TotalBoardings', 'Boardings_Daily', 'InitialBoardings', 'On', 'DailyBoard_2050'],
    'alightings_cols': ['Alightings', 'TotalAlight', 'Alightings_Daily', 'FinalAlight', 'Off', 'DailyAlight_2050'],
    'transfer_cols': ['Transfers', 'TotalTransfers', 'TransferBoardings', 'TransferAlight'],
}

print("Sheet column configurations defined:")
for region in SHEET_COLUMN_CONFIG:
    print(f"  - {region}")
print(f"  - Default (for unknown sheets)")

## Step 2.5: Load Demand Data from Excel

Load demand data from Excel file with multiple regional model sheets.
Each sheet corresponds to a different transport model (Haifa, Tel Aviv, etc.).

In [ ]:
if PART2_AVAILABLE:
    print("Step 2.5: Loading demand data from Excel...")
    
    def find_column(df_columns, possible_names):
        """Find matching column from list of possible names."""
        for name in possible_names:
            if name in df_columns:
                return name
            # Case-insensitive fallback
            for col in df_columns:
                if col.lower() == name.lower():
                    return col
        return None
    
    def process_sheet(df, sheet_name, region_name):
        """Process a single sheet and return dict of node_id -> (demand, transfers)."""
        config = SHEET_COLUMN_CONFIG.get(region_name, DEFAULT_COLUMN_CONFIG)
        
        # Find columns
        node_col = find_column(df.columns, config['node_cols'])
        boardings_col = find_column(df.columns, config['boardings_cols'])
        alightings_col = find_column(df.columns, config['alightings_cols'])
        transfer_cols = [find_column(df.columns, [tc]) for tc in config.get('transfer_cols', [])]
        transfer_cols = [tc for tc in transfer_cols if tc]
        
        if node_col is None:
            print(f"    ⚠ No node column found in '{sheet_name}'")
            return {}
        
        print(f"    Columns: node='{node_col}', boardings='{boardings_col}', alightings='{alightings_col}'")
        
        result = {}
        for _, row in df.iterrows():
            try:
                node_val = row[node_col]
                if pd.isna(node_val):
                    continue
                node_id = int(float(node_val))
                
                boardings = float(row[boardings_col]) if boardings_col and pd.notna(row.get(boardings_col)) else 0
                alightings = float(row[alightings_col]) if alightings_col and pd.notna(row.get(alightings_col)) else 0
                demand = boardings + alightings
                
                transfers = sum(float(row[tc]) for tc in transfer_cols if pd.notna(row.get(tc)))
                
                if node_id not in result:
                    result[node_id] = {'demand': demand, 'transfers': transfers}
                else:
                    result[node_id]['demand'] += demand
                    result[node_id]['transfers'] += transfers
            except (ValueError, TypeError):
                continue
        
        print(f"    ✓ Processed {len(result)} unique nodes")
        return result
    
    # Check if Excel file exists
    if not os.path.exists(DEMAND_EXCEL):
        print(f"  ✗ Demand Excel not found: {DEMAND_EXCEL}")
        print("    Update DEMAND_EXCEL path in Part 2 configuration")
        node_demand = {}
        region_demand = {}
    else:
        print(f"  Loading: {DEMAND_EXCEL}")
        xl = pd.ExcelFile(DEMAND_EXCEL)
        raw_sheet_names = xl.sheet_names
        print(f"  Found sheets: {raw_sheet_names}")
        
        node_demand = {}  # Global node -> demand mapping
        region_demand = {}  # Region -> {node -> demand} mapping
        
        for sheet in raw_sheet_names:
            # Map sheet name to region
            region_name = SHEET_NAME_MAPPING.get(sheet)
            if region_name is None:
                for key, value in SHEET_NAME_MAPPING.items():
                    if key.lower() == sheet.lower():
                        region_name = value
                        break
            
            if region_name is None:
                print(f"  ⚠ Unknown sheet '{sheet}', using default config")
                region_name = sheet
            
            try:
                df = pd.read_excel(DEMAND_EXCEL, sheet_name=sheet)
                print(f"\n  Sheet '{sheet}' → Region '{region_name}': {len(df)} rows")
                
                sheet_demand = process_sheet(df, sheet, region_name)
                
                # Store by region
                if region_name not in region_demand:
                    region_demand[region_name] = {}
                for node_id, data in sheet_demand.items():
                    if node_id not in region_demand[region_name]:
                        region_demand[region_name][node_id] = data
                    else:
                        region_demand[region_name][node_id]['demand'] += data['demand']
                        region_demand[region_name][node_id]['transfers'] += data['transfers']
                
                # Also store globally
                for node_id, data in sheet_demand.items():
                    if node_id not in node_demand:
                        node_demand[node_id] = data
                    else:
                        node_demand[node_id]['demand'] += data['demand']
                        node_demand[node_id]['transfers'] += data['transfers']
            
            except Exception as e:
                print(f"    ✗ Error: {e}")
        
        print(f"\n✓ Loaded demand for {len(node_demand)} unique nodes across all regions")
        print(f"  Regions with data: {list(region_demand.keys())}")
else:
    print("Step 2.5: Skipped (Part 2 not available)")

## Step 2.6: Match Demand to Hubs by Area

For each hub, sum demand from all its nodes using area-based matching:
1. Get hub's area (from Step 2.3)
2. Map area to region using AREA_TO_REGION
3. Look up node demand from that region's model
4. Sum demand for all nodes in the hub group

In [ ]:
import ast

if PART2_AVAILABLE and len(node_demand) > 0:
    print("Step 2.6: Matching demand to hubs by area...")
    
    matched_hubs = 0
    total_nodes_checked = 0
    total_nodes_matched = 0
    
    for idx, row in gdf_demand.iterrows():
        # Get list of nodes in this hub
        nodes_in_group = row.get('node', [])
        
        # Handle string representation of list from CSV
        if isinstance(nodes_in_group, str):
            try:
                nodes_in_group = ast.literal_eval(nodes_in_group)
            except (ValueError, SyntaxError):
                nodes_in_group = [nodes_in_group]
        
        if not isinstance(nodes_in_group, list):
            nodes_in_group = [nodes_in_group]
        
        # Get hub's area and map to region
        hub_area = row.get('area', None)
        hub_region = AREA_TO_REGION.get(hub_area) if hub_area else None
        
        # Sum demand from all nodes
        total_demand = 0
        total_transfers = 0
        nodes_matched = 0
        
        for node in nodes_in_group:
            total_nodes_checked += 1
            try:
                node = int(node)
            except (ValueError, TypeError):
                continue
            
            matched = False
            
            # Try area-based matching first
            if hub_region and hub_region in region_demand:
                if node in region_demand[hub_region]:
                    total_demand += region_demand[hub_region][node]['demand']
                    total_transfers += region_demand[hub_region][node]['transfers']
                    nodes_matched += 1
                    total_nodes_matched += 1
                    matched = True
            
            # Fall back to global matching
            if not matched and node in node_demand:
                total_demand += node_demand[node]['demand']
                total_transfers += node_demand[node]['transfers']
                nodes_matched += 1
                total_nodes_matched += 1
        
        # Assign to hub
        gdf_demand.at[idx, 'TotalDemand'] = total_demand
        gdf_demand.at[idx, 'TotalTransfers'] = total_transfers
        
        if nodes_matched > 0:
            matched_hubs += 1
    
    # Apply overlay models (Hadera, Haifa Metronit)
    overlay_regions = ['Hadera', 'Haifa Metronit']
    for overlay_region in overlay_regions:
        if overlay_region in region_demand:
            overlay_matched = 0
            for idx, row in gdf_demand.iterrows():
                nodes_in_group = row.get('node', [])
                if isinstance(nodes_in_group, str):
                    try:
                        nodes_in_group = ast.literal_eval(nodes_in_group)
                    except:
                        nodes_in_group = [nodes_in_group]
                if not isinstance(nodes_in_group, list):
                    nodes_in_group = [nodes_in_group]
                
                for node in nodes_in_group:
                    try:
                        node = int(node)
                    except:
                        continue
                    if node in region_demand[overlay_region]:
                        gdf_demand.at[idx, 'TotalDemand'] += region_demand[overlay_region][node]['demand']
                        gdf_demand.at[idx, 'TotalTransfers'] += region_demand[overlay_region][node]['transfers']
                        overlay_matched += 1
            if overlay_matched > 0:
                print(f"  Applied {overlay_region} overlay: {overlay_matched} nodes updated")
    
    # Summary
    print("\n  DEMAND MATCHING SUMMARY:")
    print(f"  ─────────────────────────")
    print(f"  Hubs processed: {len(gdf_demand)}")
    print(f"  Hubs with demand: {matched_hubs} ({matched_hubs/len(gdf_demand)*100:.1f}%)")
    print(f"  Nodes checked: {total_nodes_checked}")
    print(f"  Nodes matched: {total_nodes_matched} ({total_nodes_matched/max(total_nodes_checked,1)*100:.1f}%)")
    print(f"  Total demand: {gdf_demand['TotalDemand'].sum():,.0f}")
    print(f"  Total transfers: {gdf_demand['TotalTransfers'].sum():,.0f}")
    print(f"  Demand range: {gdf_demand['TotalDemand'].min():,.0f} - {gdf_demand['TotalDemand'].max():,.0f}")
    
    if total_nodes_matched == 0:
        print("\n  ⚠ WARNING: No nodes matched!")
        print("    Check that node IDs match between hub data and demand file")
    
    print(f"\n✓ Step 2.6 complete!")
    
elif PART2_AVAILABLE:
    print("Step 2.6: Skipped (no demand data loaded)")
else:
    print("Step 2.6: Skipped (Part 2 not available)")

## Step 2.6.1: Load Manual Demand Updates from CSV (Optional)

Load demand corrections from CSV file if available.

**CSV Format:**
- **node**: Node ID (integer)
- **area**: Region name (e.g., 'Tel Aviv', 'Haifa', 'National')
- **total_demand**: Updated total demand value
- **total_transfers**: Updated total transfers value

If the CSV file doesn't exist, this step is skipped.

In [ ]:
if PART2_AVAILABLE:
    print("Step 2.6.1: Loading manual demand updates from CSV...")
    
    manual_updates_applied = 0
    
    if MANUAL_DEMAND_UPDATES_CSV and os.path.exists(MANUAL_DEMAND_UPDATES_CSV):
        try:
            # Load CSV with demand updates
            updates_df = pd.read_csv(MANUAL_DEMAND_UPDATES_CSV)
            print(f"  ✓ Loaded {len(updates_df)} demand updates from CSV")
            print(f"    File: {MANUAL_DEMAND_UPDATES_CSV}")
            
            # Validate required columns
            required_cols = ['node', 'total_demand', 'total_transfers']
            missing_cols = [col for col in required_cols if col not in updates_df.columns]
            
            if missing_cols:
                print(f"  ⚠ Missing required columns: {missing_cols}")
                print(f"    Required: {required_cols}")
                print(f"    Found: {updates_df.columns.tolist()}")
                print(f"    Skipping CSV updates")
            else:
                # Apply updates
                for _, update_row in updates_df.iterrows():
                    node_id = int(update_row['node'])
                    demand = update_row['total_demand']
                    transfers = update_row['total_transfers']
                    area = update_row.get('area', 'Unknown')
                    
                    # Find matching nodes in gdf_demand
                    mask = gdf_demand['node'] == node_id
                    
                    if mask.any():
                        gdf_demand.loc[mask, 'TotalDemand'] = demand
                        gdf_demand.loc[mask, 'TotalTransfers'] = transfers
                        manual_updates_applied += 1
                        print(f"  ✓ Updated node {node_id} ({area}): {demand:,.0f} demand, {transfers:,.0f} transfers")
                    else:
                        print(f"  ⚠ Node {node_id} ({area}) not found in data - skipped")
                
                print(f"\n  Total updates from CSV: {manual_updates_applied}")
                
        except Exception as e:
            print(f"  ⚠ Error loading CSV: {e}")
            print(f"    Proceeding with default demand data")
    else:
        if not MANUAL_DEMAND_UPDATES_CSV:
            print(f"  CSV path not configured (MANUAL_DEMAND_UPDATES_CSV)")
        else:
            print(f"  CSV file not found: {MANUAL_DEMAND_UPDATES_CSV}")
        print(f"  Proceeding with default demand data")
else:
    print("Skipping CSV demand updates (Part 2 not available)")

## Step 2.6.2: Apply Hardcoded Demand Updates from National Model

Apply specific node-level demand corrections from National Model data.

**Note:** Index-based updates (for nodes with multiple modes like Netanya, Modiin) are **NOT** applied here
because group IDs may have changed. Use the CSV update method (Step 2.6.1) for those updates instead.

In [ ]:
if PART2_AVAILABLE:
    print("Step 2.6.2: Applying hardcoded demand updates from National Model...")
    
    updated_count = 0
    
    # Node-based updates (safe to apply)
    node_updates = {
        400424: (64985, 43032, 'Moshe Dayan (Rishon)'),
        400021: (23083, 10140, 'Netanya Sapir'),
        400030: (14518, 6101, 'Beit Yehoshua Rail'),
        511246: (13601, 6101, 'Beit Yehoshua LRT'),
    }
    
    for node_id, (demand, transfers, name) in node_updates.items():
        mask = gdf_demand['node'] == node_id
        if mask.any():
            gdf_demand.loc[mask, 'TotalDemand'] = demand
            gdf_demand.loc[mask, 'TotalTransfers'] = transfers
            updated_count += 1
            print(f"  ✓ Updated {name} (node {node_id}): {demand:,.0f} demand, {transfers:,.0f} transfers")
        else:
            print(f"  ⚠ {name} (node {node_id}) not found in data - skipped")
    
    # Index-based updates - SKIPPED with warning
    print("\n  ⚠ SKIPPED INDEX-BASED UPDATES:")
    print("    The following updates from the old code are NOT applied because")
    print("    group/index IDs may have changed in the new grouping:")
    print("")
    print("    - Netanya (indices 1057, 1058): node 400020")
    print("      Original: 108,409 demand, 84,490 transfers")
    print("")
    print("    - Modiin Merkaz (indices 421, 422): node 400470")
    print("      Original: 40,628 demand, 0 transfers")
    print("")
    print("    - Modiin West (indices 419, 420): node 400460")
    print("      Original: 41,000 demand, 12,133 transfers")
    print("")
    print("    To apply these updates, create a CSV file with these nodes and use")
    print("    Step 2.6.1 (CSV-based updates) instead.")
    print("")
    
    print(f"  Total hardcoded updates applied: {updated_count}")
    print(f"  Total demand after hardcoded updates: {gdf_demand['TotalDemand'].sum():,.0f}")
    print(f"  Total transfers after hardcoded updates: {gdf_demand['TotalTransfers'].sum():,.0f}")
else:
    print("Skipping hardcoded demand updates (Part 2 not available)")

## Step 2.6.3: Update Shefaim LRT Stop

Apply specific demand correction for Shefaim LRT station (node 511248).

In [ ]:
if PART2_AVAILABLE:
    print("Step 2.6.3: Updating Shefaim LRT stop demand...")

    # Update Shefaim LRT station
    shefaim_node = 511248
    shefaim_demand = 255.3

    mask = gdf_demand['node'] == shefaim_node
    if mask.any():
        gdf_demand.loc[mask, 'TotalDemand'] = shefaim_demand
        print(f"  ✓ Updated Shefaim LRT (node {shefaim_node}): {shefaim_demand} demand")
    else:
        print(f"  ⚠ Shefaim LRT node {shefaim_node} not found in data")
else:
    print("Skipping Shefaim update (Part 2 not available)")

## Step 2.7: Create Grouped Hubs with Demand

Aggregate individual H3 hexagons into hub groups, summing demand.

In [ ]:
if PART2_AVAILABLE:
    print("Step 2.7: Creating grouped hubs with demand...")
    
    # Build aggregation functions
    def flatten_list(x):
        """Flatten lists and collect unique values."""
        all_items = set()
        for item in x:
            if isinstance(item, list):
                all_items.update(item)
            elif pd.notna(item):
                all_items.add(item)
        return list(all_items)
    
    def clean_nodes(x):
        """Clean and flatten node lists."""
        all_nodes = set()
        for item in x:
            if isinstance(item, list):
                for node in item:
                    if pd.notna(node):
                        try:
                            all_nodes.add(int(node))
                        except:
                            all_nodes.add(node)
            elif pd.notna(item):
                try:
                    all_nodes.add(int(item))
                except:
                    all_nodes.add(item)
        return list(all_nodes)
    
    # FIX: Custom aggregation functions for string columns to prevent float64 conversion
    def first_string_value(x):
        """Get first non-null string value, or 'Unknown' if all null.
        This prevents pandas from converting string columns to float64."""
        for val in x:
            if pd.notna(val) and val != '' and val != []:
                # Return as string (not list)
                return str(val) if not isinstance(val, list) else val
        return 'Unknown'
    
    def first_list_value(x):
        """Get first non-null list value, or ['Unknown'] if all null.
        This preserves list columns like 'location'."""
        for val in x:
            if isinstance(val, list) and len(val) > 0:
                return val
            elif pd.notna(val) and val != '':
                return [str(val)]
        return ['Unknown']
    
    # Build aggregation dictionary
    agg_dict = {}
    
    if 'h3_index' in gdf_demand.columns:
        agg_dict['h3_index'] = flatten_list
    if 'node' in gdf_demand.columns:
        agg_dict['node'] = clean_nodes
    if 'Mode_Planned' in gdf_demand.columns:
        agg_dict['Mode_Planned'] = flatten_list
    if 'Line_Nunique' in gdf_demand.columns:
        agg_dict['Line_Nunique'] = 'sum'
    if 'Line_Unique' in gdf_demand.columns:
        agg_dict['Line_Unique'] = flatten_list
    if 'TotalDemand' in gdf_demand.columns:
        agg_dict['TotalDemand'] = 'sum'
    if 'TotalTransfers' in gdf_demand.columns:
        agg_dict['TotalTransfers'] = 'sum'
    
    # FIX: Use custom aggregation for string columns (NOT 'first' which converts to float64)
    for col in ['address', 'area', 'district', 'metro_area']:
        if col in gdf_demand.columns:
            agg_dict[col] = first_string_value
    
    # FIX: Use list-preserving aggregation for location column
    if 'location' in gdf_demand.columns:
        agg_dict['location'] = first_list_value
    
    # Add mode-specific line columns to aggregation (sum across hexes)
    MODE_LINE_COLS = [
        'BRT Lines', 'Cable Line Lines', 'Funicular Lines',
        'HighSpeed Rail Lines', 'Interurban Rail Lines', 'LRT Lines',
        'Metro Lines', 'Suburban Rail Lines'
    ]
    for col in MODE_LINE_COLS:
        if col in gdf_demand.columns:
            agg_dict[col] = 'sum'
    
    # Group by 'group' column
    grouped = gdf_demand.groupby('group').agg(agg_dict).reset_index()
    
    # Calculate number of modes - FIXED: count mode-specific line columns > 0
    # MODE LINE COLUMNS to check
    MODE_LINE_COLS = [
        'BRT Lines', 'Cable Line Lines', 'Funicular Lines',
        'HighSpeed Rail Lines', 'Interurban Rail Lines', 'LRT Lines',
        'Metro Lines', 'Suburban Rail Lines'
    ]

    def count_positive_mode_lines(row):
        """Count how many mode-specific line columns have values > 0."""
        count = 0
        for col in MODE_LINE_COLS:
            if col in row.index and pd.notna(row[col]) and row[col] > 0:
                count += 1
        return count

    grouped['Num_Modes'] = grouped.apply(count_positive_mode_lines, axis=1)
    
    # Dissolve geometries
    try:
        dissolved = gdf_demand.dissolve(by='group', as_index=False)
        grouped['geometry'] = dissolved['geometry'].values
    except Exception as e:
        print(f"  ⚠ Dissolve failed: {e}")
        grouped['geometry'] = gdf_demand.groupby('group')['geometry'].first().values
    
    # Create GeoDataFrame
    grouped_hubs = gpd.GeoDataFrame(grouped, geometry='geometry', crs=gdf_demand.crs)
    
    print(f"\n✓ Created {len(grouped_hubs)} hub groups")
    print(f"  Total demand: {grouped_hubs['TotalDemand'].sum():,.0f}")
    print(f"  Hubs with demand > 0: {(grouped_hubs['TotalDemand'] > 0).sum()}")
    
    # Show top hubs by demand
    top_hubs = grouped_hubs.nlargest(5, 'TotalDemand')[['group', 'TotalDemand', 'area']]
    print(f"\n  Top 5 hubs by demand:")
    print(top_hubs.to_string(index=False))
    

    # DIAGNOSTIC: Check area/location after aggregation
    print("\n  DIAGNOSTIC - After Step 2.7 Aggregation:")
    print(f"    grouped_hubs shape: {grouped_hubs.shape}")
    print(f"    Columns: {list(grouped_hubs.columns)}")
    print(f"    'area' column exists: {'area' in grouped_hubs.columns}")
    if 'area' in grouped_hubs.columns:
        print(f"    'area' non-null count: {grouped_hubs['area'].notna().sum()}/{len(grouped_hubs)}")
        print(f"    'area' dtype: {grouped_hubs['area'].dtype}")
        print(f"    'area' unique values (first 5): {grouped_hubs['area'].dropna().unique()[:5].tolist()}")
    print(f"    'location' column exists: {'location' in grouped_hubs.columns}")
    if 'location' in grouped_hubs.columns:
        print(f"    'location' non-null count: {grouped_hubs['location'].notna().sum()}/{len(grouped_hubs)}")
        sample_locs = grouped_hubs[grouped_hubs['location'].notna()]['location'].head(3).tolist()
        print(f"    'location' sample values: {sample_locs}")

    print(f"\n✓ Step 2.7 complete!")
else:
    print("Step 2.7: Skipped (Part 2 not available)")



## Step 2.7.2: Verify Scoring Columns

**DIAGNOSTIC STEP** - Verify that required columns for scoring exist and have valid values.


In [ ]:
if PART2_AVAILABLE:
    print("Step 2.7.2: Verifying scoring columns...")
    
    # Define expected columns
    MODE_LINE_COLS = [
        'BRT Lines', 'Cable Line Lines', 'Funicular Lines',
        'HighSpeed Rail Lines', 'Interurban Rail Lines', 'LRT Lines',
        'Metro Lines', 'Suburban Rail Lines'
    ]
    
    # Check mode columns
    print("\n  Mode-specific line columns:")
    mode_cols_present = [col for col in MODE_LINE_COLS if col in grouped_hubs.columns]
    print(f"    Present: {len(mode_cols_present)}/{len(MODE_LINE_COLS)}")
    
    if mode_cols_present:
        for col in mode_cols_present:
            nonzero = (grouped_hubs[col] > 0).sum()
            total = grouped_hubs[col].sum()
            if nonzero > 0:
                print(f"      {col}: {nonzero} hubs, {total:.0f} total lines")
    else:
        print("    ⚠ WARNING: No mode-specific columns found!")
    
    # Check location column
    print("\n  Location column:")
    if 'location' in grouped_hubs.columns:
        print("    ✓ Location column present")
        location_values = grouped_hubs['location'].apply(
            lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x
        ).value_counts()
        print("    Distribution:")
        for loc, count in location_values.items():
            print(f"      {loc}: {count}")
    else:
        print("    ⚠ WARNING: Location column not found!")
    
    # Check Num_Modes
    print("\n  Num_Modes column:")
    if 'Num_Modes' in grouped_hubs.columns:
        print("    ✓ Num_Modes column present")
        num_modes_dist = grouped_hubs['Num_Modes'].value_counts().sort_index()
        print("    Distribution:")
        for modes, count in num_modes_dist.items():
            print(f"      {modes} modes: {count} hubs")
        
        if (grouped_hubs['Num_Modes'] == 0).all():
            print("    ⚠ WARNING: All Num_Modes values are 0!")
    else:
        print("    ⚠ WARNING: Num_Modes column not found!")
    
    # Expected impact on scoring
    print("\n  Expected scoring impact:")
    print("    - RegionLocation score will now vary by area and location")
    print("    - Mode score will reflect mode diversity and line counts")
    print("    - Num_Modes will enable proper multi-modal hub identification")
    
    print("\n✓ Step 2.7.2 complete!")
else:
    print("Step 2.7.2: Skipped (Part 2 not available)")


## Step 2.7.1: Add Bus Terminal Data

Spatially join bus terminal data to add `term_type` and `id` columns for scoring.

In [ ]:
if PART2_AVAILABLE:
    print("Step 2.7.1: Adding bus terminal data via spatial join...")
    
    # Check if bus terminal shapefile is configured and exists
    if BUS_TERMINALS_SHP and os.path.exists(BUS_TERMINALS_SHP):
        try:
            # Load bus terminals
            terminals_gdf = gpd.read_file(BUS_TERMINALS_SHP, encoding='utf-8')
            print(f"  ✓ Loaded {len(terminals_gdf)} bus terminals from {BUS_TERMINALS_SHP}")
            
            # Ensure both are in same CRS
            if grouped_hubs.crs != terminals_gdf.crs:
                print(f"  Converting terminals CRS from {terminals_gdf.crs} to {grouped_hubs.crs}")
                terminals_gdf = terminals_gdf.to_crs(grouped_hubs.crs)
            
            # Create buffer around bus terminals (200m) for proximity matching
            terminals_buffered = terminals_gdf.copy()
            if grouped_hubs.crs.is_geographic:
                # Convert to metric CRS for buffering
                temp_crs = 'EPSG:2039'  # ITM Israel
                terminals_buffered = terminals_buffered.to_crs(temp_crs)
                terminals_buffered['geometry'] = terminals_buffered.buffer(200)  # 200m
                terminals_buffered = terminals_buffered.to_crs(grouped_hubs.crs)
            else:
                terminals_buffered['geometry'] = terminals_buffered.buffer(200)
            
            # Select relevant columns from terminals (adapt based on your shapefile)
            terminal_cols = [col for col in terminals_gdf.columns if col in ['id', 'term_type', 'type', 'name', 'terminal_type']]
            if 'geometry' not in terminal_cols:
                terminal_cols.append('geometry')
            
            # Try to identify the terminal type column
            type_col = None
            for col in ['term_type', 'type', 'terminal_type']:
                if col in terminals_buffered.columns:
                    type_col = col
                    break
            
            if type_col:
                # Spatial join (left join - keep all hubs)
                grouped_hubs = gpd.sjoin(
                    grouped_hubs,
                    terminals_buffered[[type_col, 'geometry']],
                    how='left',
                    predicate='intersects'
                )
                
                # Rename to standard 'term_type' if needed
                if type_col != 'term_type':
                    grouped_hubs = grouped_hubs.rename(columns={type_col: 'term_type'})
                
                # Drop index_right column from sjoin
                if 'index_right' in grouped_hubs.columns:
                    grouped_hubs = grouped_hubs.drop(columns=['index_right'])
                
                print(f"  ✓ Spatial join complete")
                print(f"  Hubs with terminals: {grouped_hubs['term_type'].notna().sum()} of {len(grouped_hubs)}")
                
                if 'term_type' in grouped_hubs.columns:
                    term_dist = grouped_hubs['term_type'].value_counts()
                    print(f"  Terminal types: {term_dist.to_dict()}")
            else:
                print(f"  ⚠ Could not find terminal type column in shapefile")
                print(f"    Available columns: {terminals_gdf.columns.tolist()}")
                grouped_hubs['term_type'] = None
                
        except Exception as e:
            print(f"  ⚠ Error loading bus terminals: {e}")
            print(f"    Proceeding without terminal data")
            grouped_hubs['term_type'] = None
    else:
        if not BUS_TERMINALS_SHP:
            print(f"  Bus terminals path not configured (set BUS_TERMINALS_SHP in config)")
        else:
            print(f"  Bus terminals file not found: {BUS_TERMINALS_SHP}")
        print(f"  Proceeding without terminal data")
        grouped_hubs['term_type'] = None
    
    # Create bus_terminal score column
    def bus_terminal_score(term_type):
        """Convert terminal type to score."""
        if pd.isna(term_type):
            return 0
        term_type = str(term_type).strip()
        if 'חניון לילה' in term_type:
            return 1
        elif 'מסוף קטן' in term_type or 'מסוף בינוני' in term_type:
            return 2
        elif 'מסוף גדול' in term_type or 'מתקן משולב' in term_type:
            return 3
        return 0
    
    grouped_hubs['bus_terminal'] = grouped_hubs['term_type'].apply(bus_terminal_score)
    print(f"  ✓ Created bus_terminal score column")
    print(f"    Score distribution: {grouped_hubs['bus_terminal'].value_counts().sort_index().to_dict()}")
else:
    print("Skipping bus terminal spatial join (Part 2 not available)")

## Step 2.8: Export Grouped Hubs with Demand

In [ ]:
if PART2_AVAILABLE:
    print("Step 2.8: Exporting grouped hubs with demand...")
    
    # Ensure output directory exists
    os.makedirs(os.path.dirname(OUTPUT_GROUPED_HUBS) if os.path.dirname(OUTPUT_GROUPED_HUBS) else '.', exist_ok=True)
    
    # Export ungrouped data with demand
    if OUTPUT_HUBS_WITH_DEMAND:
        export_ungrouped = gdf_demand.copy()
        export_ungrouped['geometry'] = export_ungrouped['geometry'].apply(lambda x: x.wkt if x else None)
        export_ungrouped.to_csv(OUTPUT_HUBS_WITH_DEMAND, index=False, encoding='utf-8-sig')
        print(f"  ✓ Ungrouped hubs: {OUTPUT_HUBS_WITH_DEMAND}")
    
    # Export grouped data
    export_grouped = grouped_hubs.copy()

    # DIAGNOSTIC: Check columns before CSV export
    print("\n  DIAGNOSTIC - Before Step 2.8 CSV Export:")
    print(f"    Columns to export: {list(export_grouped.columns)}")
    print(f"    'area' in columns: {'area' in export_grouped.columns}")
    print(f"    'location' in columns: {'location' in export_grouped.columns}")
    if 'area' in export_grouped.columns:
        print(f"    'area' sample: {export_grouped['area'].head(3).tolist()}")
    if 'location' in export_grouped.columns:
        print(f"    'location' sample: {export_grouped['location'].head(3).tolist()}")

    export_grouped['geometry'] = export_grouped['geometry'].apply(lambda x: x.wkt if x else None)
    export_grouped.to_csv(OUTPUT_GROUPED_HUBS, index=False, encoding='utf-8-sig')
    print(f"  ✓ Grouped hubs: {OUTPUT_GROUPED_HUBS}")
    
    print(f"\n" + "="*80)
    print("PART 2 COMPLETE - DEMAND DATA PROCESSING")
    print("="*80)
    print(f"\nResults:")
    print(f"  Hub groups: {len(grouped_hubs)}")
    print(f"  Total demand: {grouped_hubs['TotalDemand'].sum():,.0f}")
    print(f"  Hubs with demand: {(grouped_hubs['TotalDemand'] > 0).sum()}")
    print(f"\nOutput files:")
    print(f"  {OUTPUT_GROUPED_HUBS}")
    if OUTPUT_HUBS_WITH_DEMAND:
        print(f"  {OUTPUT_HUBS_WITH_DEMAND}")
    print(f"\nReady for Part 3: Influence Area Processing")
else:
    print("Step 2.8: Skipped (Part 2 not available)")

---
# PART 3: INFLUENCE AREA PROCESSING (OPTIONAL)
---

This part adds demographic data (population, employment) based on buffer zones around each hub.

**What it does:**
- Creates 3 concentric buffer zones (0-600m, 600-1000m, 1000-1200m)
- Calculates population and employment from TAZ (Traffic Analysis Zones)
- Uses proportional allocation based on area overlap
- Identifies hubs near bus terminals

**Requirements:**
- `influence_area_processor.py` module
- TAZ shapefile with `POP_2050` and `EMPL_2050` columns
- (Optional) Bus terminals shapefile

**Expected runtime:** ~2-3 minutes for 1000-2000 hubs

## Step 3.1: Configure Paths for Part 3

In [ ]:
# ============================================================================
# PART 3 CONFIGURATION - INFLUENCE AREA PROCESSING
# ============================================================================
# File paths are configured in the MASTER CONFIGURATION section above.
# This section just displays what will be used.

# Input: Grouped hubs from Part 2 (or Part 1 if Part 2 was skipped)
if PART2_AVAILABLE:
    INPUT_FOR_INFLUENCE = OUTPUT_GROUPED_HUBS
else:
    INPUT_FOR_INFLUENCE = OUTPUT_H3_HEXAGONS

print("Part 3 Configuration:")
print(f"  Input hubs: {INPUT_FOR_INFLUENCE}")
print(f"  TAZ Shapefile: {TAZ_SHAPEFILE}")
print(f"  Bus Terminals: {BUS_TERMINALS_SHP if BUS_TERMINALS_SHP else 'Not provided (optional)'}")
print(f"  Output CSV: {OUTPUT_FINAL}")
print(f"  Output Excel: {OUTPUT_FINAL_EXCEL}")
print(f"  Buffer zones: {BUFFER_ZONES}")


## Step 3.2: Load Influence Area Processor Module

In [ ]:
# Import influence area processor
# NOTE: This requires influence_area_processor.py to be in the same directory
# or in your Python path

try:
    from influence_area_processor import InfluenceAreaProcessor
    print("✓ Influence area processor module loaded successfully!")
    PART3_AVAILABLE = True
except ImportError as e:
    print("✗ Could not load influence_area_processor module")
    print(f"  Error: {e}")
    print(f"\n  To run Part 3, ensure influence_area_processor.py is available.")
    print(f"  You can continue with results from Parts 1 and 2.")
    PART3_AVAILABLE = False

## Step 3.3: Check TAZ Data Availability

In [ ]:
import os

if PART3_AVAILABLE:
    print("Checking data availability...")
    
    # Check TAZ shapefile
    if os.path.exists(TAZ_SHAPEFILE):
        print(f"  ✓ TAZ shapefile found: {TAZ_SHAPEFILE}")
        TAZ_AVAILABLE = True
    else:
        print(f"  ✗ TAZ shapefile not found: {TAZ_SHAPEFILE}")
        print(f"    Update the path in Step 3.1 or skip Part 3")
        TAZ_AVAILABLE = False
    
    # Check bus terminals (optional)
    if BUS_TERMINALS_SHP and os.path.exists(BUS_TERMINALS_SHP):
        print(f"  ✓ Bus terminals found: {BUS_TERMINALS_SHP}")
        TERMINALS_AVAILABLE = True
    elif BUS_TERMINALS_SHP:
        print(f"  ⚠ Bus terminals not found: {BUS_TERMINALS_SHP}")
        print(f"    Will proceed without terminal proximity data")
        TERMINALS_AVAILABLE = False
    else:
        print(f"  ⚠ Bus terminals not configured (optional)")
        TERMINALS_AVAILABLE = False
    
    # Check input hubs
    if os.path.exists(INPUT_FOR_INFLUENCE):
        print(f"  ✓ Input hubs found: {INPUT_FOR_INFLUENCE}")
    else:
        print(f"  ✗ Input hubs not found: {INPUT_FOR_INFLUENCE}")
        print(f"    Make sure Parts 1 and 2 completed successfully")
        TAZ_AVAILABLE = False
else:
    print("Skipping data check (influence_area_processor not available)")
    TAZ_AVAILABLE = False

## Step 3.4: Initialize Processor and Run Pipeline

In [ ]:
if PART3_AVAILABLE and TAZ_AVAILABLE:
    print("Step 3.4: Running influence area processing pipeline...")
    print("="*80)
    
    # Initialize processor
    processor = InfluenceAreaProcessor()
    
    # Set custom buffer zones if specified
    processor.buffer_zones = BUFFER_ZONES
    
    # Run full pipeline
    try:
        final_gdf = processor.process_full_pipeline(
            hubs_csv=INPUT_FOR_INFLUENCE,
            taz_shp=TAZ_SHAPEFILE,
            terminals_shp=BUS_TERMINALS_SHP if TERMINALS_AVAILABLE else None,
            output_csv=OUTPUT_FINAL
        )
        
        print(f"\n✓ Part 3 complete!")
        print(f"  Final dataset: {len(final_gdf)} hub groups")
        print(f"  Output file: {OUTPUT_FINAL}")
        
        PART3_SUCCESS = True
        
    except Exception as e:
        print(f"\n✗ Error during influence area processing:")
        print(f"  {e}")
        print(f"\n  You can still use the output from Parts 1 and 2.")
        PART3_SUCCESS = False
        final_gdf = None
else:
    if not PART3_AVAILABLE:
        print("Step 3.4: Skipped (influence_area_processor not available)")
    elif not TAZ_AVAILABLE:
        print("Step 3.4: Skipped (TAZ data not available)")
    
    print("\nParts 1 and 2 output is available:")
    if PART2_AVAILABLE:
        print(f"  {OUTPUT_GROUPED_HUBS}")
    else:
        print(f"  {OUTPUT_H3_HEXAGONS}")
    
    PART3_SUCCESS = False
    final_gdf = None

## Step 3.5: Explore Results (if Part 3 completed)

In [ ]:
if PART3_SUCCESS and final_gdf is not None:
    print("Part 3 Results Summary")
    print("="*80)
    
    # Show columns added
    influence_cols = [col for col in final_gdf.columns if 'pop_' in col or 'emp_' in col or 'terminal' in col]
    print(f"\nColumns added by influence area processing:")
    for col in influence_cols:
        print(f"  - {col}")
    
    # Statistics
    print(f"\nPopulation Statistics:")
    if 'total_pop_influence' in final_gdf.columns:
        print(f"  Total population in influence areas: {final_gdf['total_pop_influence'].sum():,.0f}")
        print(f"  Average per hub: {final_gdf['total_pop_influence'].mean():,.0f}")
        print(f"  Max in single hub: {final_gdf['total_pop_influence'].max():,.0f}")
    
    print(f"\nEmployment Statistics:")
    if 'total_emp_influence' in final_gdf.columns:
        print(f"  Total employment in influence areas: {final_gdf['total_emp_influence'].sum():,.0f}")
        print(f"  Average per hub: {final_gdf['total_emp_influence'].mean():,.0f}")
        print(f"  Max in single hub: {final_gdf['total_emp_influence'].max():,.0f}")
    
    # Bus terminal proximity
    if 'near_bus_terminal' in final_gdf.columns:
        near_terminals = final_gdf['near_bus_terminal'].sum()
        print(f"\nBus Terminal Proximity:")
        print(f"  Hubs within 200m of terminal: {near_terminals}")
        print(f"  Percentage: {near_terminals/len(final_gdf)*100:.1f}%")
    
    # Preview data
    print(f"\nFirst 3 records:")
    display_cols = ['group', 'total_pop_influence', 'total_emp_influence']
    if 'TotalDemand' in final_gdf.columns:
        display_cols.append('TotalDemand')
    if 'near_bus_terminal' in final_gdf.columns:
        display_cols.append('near_bus_terminal')
    
    print(final_gdf[display_cols].head(3))
else:
    print("Part 3 did not complete - no results to display")

## Step 3.6: Export to Excel with Multiple Sheets (Optional)

In [ ]:
if PART3_SUCCESS and final_gdf is not None:
    print("Exporting to Excel with analysis sheets...")
    
    try:
        with pd.ExcelWriter(OUTPUT_FINAL_EXCEL, engine='openpyxl') as writer:
            # Main data sheet
            export_df = final_gdf.copy()
            if 'geometry' in export_df.columns:
                export_df['geometry'] = export_df['geometry'].apply(lambda x: x.wkt if x else None)
            export_df.to_excel(writer, sheet_name='Hub Groups', index=False)
            
            # Top hubs by population
            if 'total_pop_influence' in final_gdf.columns:
                top_pop = final_gdf.nlargest(20, 'total_pop_influence')[[
                    'group', 'total_pop_influence', 'total_emp_influence', 'address'
                ]].copy()
                top_pop.to_excel(writer, sheet_name='Top by Population', index=False)
            
            # Top hubs by employment
            if 'total_emp_influence' in final_gdf.columns:
                top_emp = final_gdf.nlargest(20, 'total_emp_influence')[[
                    'group', 'total_pop_influence', 'total_emp_influence', 'address'
                ]].copy()
                top_emp.to_excel(writer, sheet_name='Top by Employment', index=False)
            
            # Combined score (if demand data available)
            if 'TotalDemand' in final_gdf.columns and 'total_pop_influence' in final_gdf.columns:
                combined = final_gdf.copy()
                # Normalize both metrics to 0-1 scale
                combined['demand_norm'] = (combined['TotalDemand'] - combined['TotalDemand'].min()) / \
                                          (combined['TotalDemand'].max() - combined['TotalDemand'].min())
                combined['pop_norm'] = (combined['total_pop_influence'] - combined['total_pop_influence'].min()) / \
                                       (combined['total_pop_influence'].max() - combined['total_pop_influence'].min())
                combined['combined_score'] = (combined['demand_norm'] + combined['pop_norm']) / 2
                
                top_combined = combined.nlargest(20, 'combined_score')[[
                    'group', 'TotalDemand', 'total_pop_influence', 'combined_score', 'address'
                ]].copy()
                top_combined.to_excel(writer, sheet_name='Top by Combined Score', index=False)
            
            # Zone statistics
            zone_stats = pd.DataFrame({
                'Zone': ['Zone 1 (0-600m)', 'Zone 2 (600-1000m)', 'Zone 3 (1000-1200m)', 'Total'],
                'Avg Population': [
                    final_gdf['pop_zone1'].mean() if 'pop_zone1' in final_gdf.columns else 0,
                    final_gdf['pop_zone2'].mean() if 'pop_zone2' in final_gdf.columns else 0,
                    final_gdf['pop_zone3'].mean() if 'pop_zone3' in final_gdf.columns else 0,
                    final_gdf['total_pop_influence'].mean() if 'total_pop_influence' in final_gdf.columns else 0
                ],
                'Avg Employment': [
                    final_gdf['emp_zone1'].mean() if 'emp_zone1' in final_gdf.columns else 0,
                    final_gdf['emp_zone2'].mean() if 'emp_zone2' in final_gdf.columns else 0,
                    final_gdf['emp_zone3'].mean() if 'emp_zone3' in final_gdf.columns else 0,
                    final_gdf['total_emp_influence'].mean() if 'total_emp_influence' in final_gdf.columns else 0
                ]
            })
            zone_stats.to_excel(writer, sheet_name='Zone Statistics', index=False)
        
        print(f"✓ Excel file created: {OUTPUT_FINAL_EXCEL}")
        print(f"  Sheets: Hub Groups, Top by Population, Top by Employment, Zone Statistics")
        if 'TotalDemand' in final_gdf.columns:
            print(f"          Top by Combined Score")
    
    except Exception as e:
        print(f"⚠ Could not create Excel file: {e}")
        print(f"  CSV file is still available: {OUTPUT_FINAL}")
else:
    print("Skipping Excel export (Part 3 did not complete)")

---
# PART 4: SCORING AND PRIORITIZATION
---

This part implements the hub prioritization scoring system:

**What it does:**
1. Data cleaning (convert string lists to proper lists)
2. Mode name standardization
3. Regional and location categorization
4. Mode service score calculation
5. Bus terminal scoring
6. Hub type classification (National/Regional/Local/Train Station)
7. 5 criteria scoring (all normalized 1-10 per hub type):
   - RegionLocation (geographic/strategic importance)
   - bus_terminal (bus integration)
   - score (mode service diversity)
   - TotalDemand (log-transformed)
   - PopEmp (population/employment with distance decay)
8. Monte Carlo aggregation (10,000 iterations with random weights)
9. Rankings (overall and within hub type)

**Based on:** Group_n_Filter_Hubs.ipynb and HubsScoring_vAugust2025.ipynb

**Expected runtime:** ~30 seconds for 100-200 hubs

In [ ]:
print("="*80)
print("COMPLETE PIPELINE SUMMARY")
print("="*80)

print(f"\nPart 1: H3 Hexagon Processing")
print(f"  ✓ Created {len(gdf_output_part1)} H3 hexagons")
print(f"  ✓ Identified {gdf_output_part1['group'].nunique()} proximity groups")
print(f"  ✓ Output: {OUTPUT_H3_HEXAGONS}")

if PART2_AVAILABLE:
    print(f"\nPart 2: Demand Data Processing")
    print(f"  ✓ Added daily demand data")
    print(f"  ✓ Created {len(grouped_hubs)} hub groups")
    print(f"  ✓ Total demand: {grouped_hubs['TotalDemand'].sum():,.0f}")
    print(f"  ✓ Output: {OUTPUT_GROUPED_HUBS}")
else:
    print(f"\nPart 2: Demand Data Processing")
    print(f"  ✗ Skipped")

print(f"\nPart 3: Influence Area Processing")
if PART3_SUCCESS:
    print(f"  ✓ Added population and employment data")
    print(f"  ✓ Created buffer zone statistics")
    if TERMINALS_AVAILABLE:
        print(f"  ✓ Identified terminal proximity")
    print(f"  ✓ Output CSV: {OUTPUT_FINAL}")
    print(f"  ✓ Output Excel: {OUTPUT_FINAL_EXCEL}")
elif PART3_AVAILABLE and not TAZ_AVAILABLE:
    print(f"  ✗ Skipped (TAZ data not available)")
else:
    print(f"  ✗ Skipped")

print(f"\nPart 4: Scoring and Prioritization")
if 'df_scored' in locals() and df_scored is not None:
    print(f"  ✓ Scored and ranked {len(df_scored)} hubs")
    print(f"  ✓ Hub types: {df_scored['HubType'].value_counts().to_dict()}")
    print(f"  ✓ Score range: {df_scored['Average_Simulated_Score'].min():.3f} - {df_scored['Average_Simulated_Score'].max():.3f}")
    print(f"  ✓ Monte Carlo: {MONTE_CARLO_ITERATIONS:,} iterations")
    print(f"  ✓ Output CSV: {OUTPUT_SCORED_HUBS}")
    print(f"  ✓ Output Excel: {OUTPUT_SCORED_EXCEL}")
else:
    print(f"  ✗ Not run")

print(f"\n" + "="*80)
print(f"PIPELINE COMPLETE!")
print("="*80)

# Summary of available outputs
print(f"\nAvailable Output Files:")
print(f"  1. H3 Hexagons: {OUTPUT_H3_HEXAGONS}")
if PART2_AVAILABLE:
    print(f"  2. With Demand: {OUTPUT_GROUPED_HUBS}")
if PART3_SUCCESS:
    print(f"  3. Complete Data (CSV): {OUTPUT_FINAL}")
    print(f"  4. Complete Data (Excel): {OUTPUT_FINAL_EXCEL}")
if 'df_scored' in locals() and df_scored is not None:
    print(f"  5. Scored Hubs (CSV): {OUTPUT_SCORED_HUBS}")
    print(f"  6. Scored Hubs (Excel): {OUTPUT_SCORED_EXCEL}")

print(f"\n" + "="*80)
print("Ready for analysis and visualization!")
print("="*80)

## Step 4.1: Configuration for Scoring

### Optional: AHP (Analytic Hierarchy Process) Scoring

**Expert-Driven Weighting Alternative** to Monte Carlo:
- Uses expert pairwise comparisons (Saaty scale 1-9)
- Systematic weight derivation via eigenvector method
- Consistency checking (CR < 0.10)
- Compare AHP results with Monte Carlo for validation

**Configuration**: Set `AHP_ENABLED = True` in `src/config.py` and provide expert comparisons CSV.

See `docs/AHP_SCORING_GUIDE.md` for details.


In [ ]:
# Optional: Run AHP Scoring
# Expert-driven alternative to Monte Carlo weighting

if AHP_ENABLED:
    print("AHP Scoring: ENABLED")
    print(f"  Expert CSV path: {AHP_EXPERT_CSV_PATH}\n")
    
    try:
        from pathlib import Path
        from src.scoring.ahp import run_ahp_scoring_pipeline, compare_monte_carlo_vs_ahp
        
        # Check if expert CSV exists
        if not Path(AHP_EXPERT_CSV_PATH).exists():
            print(f"⚠ AHP expert CSV not found: {AHP_EXPERT_CSV_PATH}")
            print("  To use AHP scoring:")
            print("  1. Create expert comparisons CSV (see data/ahp_expert_comparisons_TEMPLATE.csv)")
            print("  2. Update AHP_EXPERT_CSV_PATH in src/config.py")
            print("  3. Set AHP_ENABLED = True")
        else:
            # Run AHP scoring (assuming df_scored exists with Monte Carlo results)
            if 'df_scored' in globals() and 'final_score' in df_scored.columns:
                print("Running AHP scoring pipeline...\n")
                
                df_scored_ahp, ahp_diagnostics = run_ahp_scoring_pipeline(
                    df_scored,
                    expert_csv_path=str(AHP_EXPERT_CSV_PATH),
                )
                
                # Update df_scored with AHP results
                df_scored = df_scored_ahp
                
                print("\n✓ AHP Scoring complete!")
                print(f"  New columns added: ahp_score, ahp_rank")
                
                # Compare methods
                print("\nComparing Monte Carlo vs AHP:")
                comparison = compare_monte_carlo_vs_ahp(df_scored)
                print(f"  Correlation (scores): {comparison['score_correlation']:.3f}")
                print(f"  Rank agreement (top 10): {comparison['top10_overlap']}/10")
                
                # Display top hubs by AHP
                print("\nTop 10 Hubs by AHP Score:")
                top_ahp = df_scored.nlargest(10, 'ahp_score')
                for rank, (idx, row) in enumerate(top_ahp.iterrows(), 1):
                    hub_id = row.get('group', idx)
                    ahp = row['ahp_score']
                    mc = row['final_score']
                    print(f"  {rank}. Hub {hub_id}: AHP={ahp:.2f}, MC={mc:.2f}")
            else:
                print("⚠ df_scored not found or missing final_score column")
                print("  Run Monte Carlo scoring first")
    
    except ImportError as e:
        print(f"⚠ Could not import AHP module: {e}")
        print("  Ensure src/scoring/ahp.py is available")
    except Exception as e:
        print(f"⚠ AHP Scoring failed: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"AHP Scoring: DISABLED (AHP_ENABLED = {AHP_ENABLED})")
    print("  Using Monte Carlo scoring only")
    print("  To enable: Set AHP_ENABLED = True in src/config.py")


### Optional: Monte Carlo Distribution Analysis

**Extended Monte Carlo Analysis** with full distribution reporting:
- Distribution statistics per hub (mean, median, percentiles, std)
- Rank robustness metrics (mean_rank, p_top1, p_top3, p_top5)
- Visualizations: boxplots, probability charts, per-hub histograms
- CSV export for detailed analysis

This provides deeper insight into score uncertainty and ranking robustness.


In [ ]:
# Optional: Run Monte Carlo Distribution Analysis
# Provides full distribution statistics and visualizations

RUN_MC_DISTRIBUTION = True  # Set to False to skip

if RUN_MC_DISTRIBUTION:
    print("Running Monte Carlo Distribution Analysis...")
    print("This may take a few minutes...\n")
    
    try:
        from src.scoring.mc_distribution import run_mc_distribution_analysis
        
        # Prepare score matrix (assuming scored hubs are in df_scored)
        if 'df_scored' in globals():
            score_columns = [
                'activity_score', 'service_score', 'location_score',
                'pop_jobs_score', 'terminal_score'
            ]
            
            # Check all columns exist
            missing_cols = [c for c in score_columns if c not in df_scored.columns]
            if missing_cols:
                print(f"⚠ Missing score columns: {missing_cols}")
                print("  Run scoring steps first")
            else:
                # Extract score matrix
                score_matrix = df_scored[score_columns].copy()
                score_matrix.index = df_scored.index
                
                # Run distribution analysis
                mc_results = run_mc_distribution_analysis(
                    score_matrix=score_matrix,
                    output_dir=OUTPUT_DIR + '/mc_distribution',
                    n_iterations=MONTE_CARLO_ITERATIONS,
                    random_seed=MONTE_CARLO_RANDOM_SEED,
                    export_raw_scores=MC_DIST_EXPORT_RAW_SCORES,
                    create_visualizations=True,
                    top_n_for_plots=MC_DIST_TOP_N_HUBS,
                )
                
                print("\n✓ MC Distribution Analysis complete!")
                print(f"  Results saved to: {OUTPUT_DIR}/mc_distribution/")
                print(f"  Files: mc_hub_stats.csv, visualizations (PNG)")
                
                # Display top hubs by robustness
                print("\nTop 10 Most Robust Hubs (by p_top3):")
                top_robust = mc_results.combined_stats.nlargest(10, 'p_top3')
                for rank, (hub_id, row) in enumerate(top_robust.iterrows(), 1):
                    print(f"  {rank}. Hub {hub_id}: p_top3={row['p_top3']:.1%}, mean_score={row['mean_score']:.2f}")
        else:
            print("⚠ df_scored not found - run scoring steps first")
    
    except ImportError as e:
        print(f"⚠ Could not import mc_distribution module: {e}")
        print("  Ensure src/scoring/mc_distribution.py is available")
    except Exception as e:
        print(f"⚠ MC Distribution Analysis failed: {e}")
        import traceback
        traceback.print_exc()
else:
    print("MC Distribution Analysis: SKIPPED (RUN_MC_DISTRIBUTION = False)")


In [ ]:
# ============================================================================
# PART 4 CONFIGURATION - SCORING AND PRIORITIZATION
# ============================================================================
# File paths and scoring parameters are configured in the MASTER CONFIGURATION above.
# This section determines which data source to use and displays configuration.

# Input: Use output from Part 2 or Part 3 if available
if PART3_SUCCESS and final_gdf is not None:
    INPUT_FOR_SCORING = final_gdf.copy()
    print("Using Part 3 output (with influence area data)")
elif PART2_AVAILABLE:
    INPUT_FOR_SCORING = grouped_hubs.copy()
    print("Using Part 2 output (with demand data)")
else:
    INPUT_FOR_SCORING = gdf_h3.copy()
    print("Using Part 1 output (H3 hexagons only)")

print(f"\nPart 4 Configuration:")
print(f"  Input: {len(INPUT_FOR_SCORING)} hub groups")
print(f"  Output CSV: {OUTPUT_SCORED_HUBS}")
print(f"  Output Excel: {OUTPUT_SCORED_EXCEL}")
print(f"\nScoring Parameters:")
print(f"  Monte Carlo iterations: {MONTE_CARLO_ITERATIONS:,}")
print(f"  Random seed: {RANDOM_SEED}")
print(f"  Distance decay beta: {DISTANCE_DECAY_BETA}")
print(f"  Hub tiers: {TIER_NATIONAL}, {TIER_METRO}, {TIER_LOCAL}")
print(f"  Mode weights: {len(MODE_WEIGHTS)} modes configured")


## Step 4.2: Data Cleaning and Preparation

In [ ]:
print("Step 4.2: Data cleaning and preparation...")

# Make a working copy
df_scoring = INPUT_FOR_SCORING.copy()

# DIAGNOSTIC: Check area/location after CSV read for scoring
print("\n  DIAGNOSTIC - After Step 4.1 CSV Read:")
print(f"    df_scoring shape: {df_scoring.shape}")
print(f"    Columns loaded: {list(df_scoring.columns)}")
print(f"    'area' column exists: {'area' in df_scoring.columns}")
if 'area' in df_scoring.columns:
    print(f"    'area' dtype: {df_scoring['area'].dtype}")
    print(f"    'area' non-null: {df_scoring['area'].notna().sum()}/{len(df_scoring)}")
    print(f"    'area' sample: {df_scoring['area'].head(5).tolist()}")
print(f"    'location' column exists: {'location' in df_scoring.columns}")
if 'location' in df_scoring.columns:
    print(f"    'location' dtype: {df_scoring['location'].dtype}")
    print(f"    'location' non-null: {df_scoring['location'].notna().sum()}/{len(df_scoring)}")
    print(f"    'location' sample: {df_scoring['location'].head(5).tolist()}")


# Map Part 3 column names to expected scoring column names
# Part 3 creates: pop_zone1, pop_zone2, pop_zone3, emp_zone1, emp_zone2, emp_zone3
# Scoring expects: pop_0_500, pop_500_1000, pop_1000_1500, emp_0_500, emp_500_1000, emp_1000_1500
column_mapping = {
    'pop_zone1': 'pop_0_500',
    'pop_zone2': 'pop_500_1000',
    'pop_zone3': 'pop_1000_1500',
    'emp_zone1': 'emp_0_500',
    'emp_zone2': 'emp_500_1000',
    'emp_zone3': 'emp_1000_1500'
}

# Check if any of the source columns exist and rename them
columns_to_rename = {k: v for k, v in column_mapping.items() if k in df_scoring.columns}
if columns_to_rename:
    df_scoring = df_scoring.rename(columns=columns_to_rename)
    print(f"  ✓ Renamed {len(columns_to_rename)} Pop/Emp columns from Part 3 format")

# Debug: Print available columns to help diagnose Pop/Emp issues
print(f"\n  Debug - Checking for Pop/Emp columns:")
print(f"  Total columns in df_scoring: {len(df_scoring.columns)}")

# Check for various population/employment column patterns
pop_emp_patterns = ['pop_', 'emp_', 'zone', 'POP', 'EMP', 'influence']
found_cols = [col for col in df_scoring.columns if any(pattern in col for pattern in pop_emp_patterns)]

if found_cols:
    print(f"  Found {len(found_cols)} columns matching Pop/Emp patterns:")
    for col in found_cols:
        print(f"    - {col}")
else:
    print(f"  ⚠ NO columns found matching Pop/Emp patterns!")
    print(f"  Available columns: {list(df_scoring.columns)[:20]}...")


# Convert string representations to lists
def split_list_string(txt):
    """Convert string representation of list to actual list."""
    # Handle already-list case first
    if isinstance(txt, list):
        return txt
    # Handle NA/None case - check separately to avoid array comparison issues
    if pd.isna(txt):
        return []
    # Convert to string for comparison
    txt_str = str(txt)
    # Handle empty string
    if txt_str == '' or txt_str == 'nan':
        return []
    # Parse the string
    txt_str = txt_str.replace("'", "").replace("[", "").replace("]", "").replace('"', "")
    return [x.strip() for x in txt_str.split(',') if x.strip()]

# Fix Mode_Planned column
if 'Mode_Planned' in df_scoring.columns:
    df_scoring['Mode_Planned'] = df_scoring['Mode_Planned'].apply(split_list_string)

# Fix Line_Unique column
if 'Line_Unique' in df_scoring.columns:
    df_scoring['Line_Unique'] = df_scoring['Line_Unique'].apply(split_list_string)

# area should remain a string (not parsed as list)
# location should already be a list, only parse if it's a string representation
if 'area' in df_scoring.columns:
    # area should be a simple string - keep it as is
    # Just ensure it's a string type, not accidentally converted
    df_scoring['area'] = df_scoring['area'].apply(
        lambda x: str(x) if pd.notna(x) and not isinstance(x, str) else x
    )

if 'location' in df_scoring.columns:
    # location should already be a list from step 2.3
    # Only parse if it's somehow a string representation (for backwards compatibility with old CSV files)
    def ensure_location_list(val):
        # Already a list - keep it
        if isinstance(val, list):
            return val
        # NA/None - return empty list for safety
        if pd.isna(val):
            return []
        # String representation like "['גלעין']" - parse it
        val_str = str(val)
        if val_str.startswith('[') and val_str.endswith(']'):
            return split_list_string(val_str)
        # Simple string - wrap in list
        return [val_str]
    
    df_scoring['location'] = df_scoring['location'].apply(ensure_location_list)

# Fix mode names
def correct_mode_planned(modes):
    """Clean up mode names and standardize."""
    if not isinstance(modes, list):
        return []
    cleaned = []
    for mode in modes:
        mode = str(mode).strip().replace(',', '')
        if mode == 'HighSpeed':
            cleaned.append('HighSpeed Rail')
        elif mode == 'Interurban':
            cleaned.append('Interurban Rail')
        elif mode == 'Suburban':
            cleaned.append('Suburban Rail')
        elif mode != 'Rail':
            cleaned.append(mode)
    # Remove generic 'Rail' if we have specific types
    if 'Rail' in cleaned:
        specific_rails = ['HighSpeed Rail', 'Interurban Rail', 'Suburban Rail']
        if any(r in cleaned for r in specific_rails):
            cleaned = [m for m in cleaned if m != 'Rail']
    return cleaned

df_scoring['Mode_Planned'] = df_scoring['Mode_Planned'].apply(correct_mode_planned)

# Count unique lines
if 'Total_Unique_Lines' not in df_scoring.columns:
    df_scoring['Total_Unique_Lines'] = df_scoring['Line_Unique'].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )

# Add region/location categories
def get_region_category(area_list):
    """
    Map area to region category.
    0 = Tel Aviv/Center (lower priority for national equity)
    1 = Periphery (higher priority for national equity)

    Args:
        area_list: List of area strings or single string

    Returns:
        0 if Tel Aviv/Center is in the list, 1 otherwise
    """
    # Handle NA/None
    if pd.isna(area_list):
        return 1  # Default to periphery

    # Ensure it's a list
    if not isinstance(area_list, list):
        area_list = [str(area_list)]

    # Check if any area in the list is Tel Aviv/Center
    for area in area_list:
        area_str = str(area).strip()
        if any(keyword in area_str for keyword in ['תל אביב', 'Tel Aviv', 'תל-אביב', 'מרכז', 'Center']):
            return 0

    # If no Tel Aviv/Center found, it's periphery
    return 1

def get_location_category(location_list):
    """
    Map location to metropolitan position category.
    3 = Core (גלעין)
    2 = Ring (טבעת)
    1 = Periphery / Other

    Args:
        location_list: List of location strings or single string

    Returns:
        Highest priority location (Core > Ring > Periphery)
    """
    # Handle NA/None
    if pd.isna(location_list):
        return 1  # Default to periphery

    # Ensure it's a list
    if not isinstance(location_list, list):
        location_list = [str(location_list)]

    # Find the highest priority location in the list
    max_category = 1  # Default to periphery

    for location in location_list:
        location_str = str(location).strip()
        if 'גלעין' in location_str or 'Core' in location_str:
            max_category = max(max_category, 3)
        elif 'טבעת' in location_str or 'Ring' in location_str:
            max_category = max(max_category, 2)

    return max_category

if 'area' in df_scoring.columns:
    df_scoring['Region_category'] = df_scoring['area'].apply(get_region_category)
else:
    df_scoring['Region_category'] = 1

if 'location' in df_scoring.columns:
    df_scoring['Location_category'] = df_scoring['location'].apply(get_location_category)
else:
    df_scoring['Location_category'] = 1

df_scoring['RegionLocation'] = df_scoring['Region_category'] * df_scoring['Location_category']

print(f"✓ Data cleaned and prepared")
print(f"  Rows: {len(df_scoring)}")
print(f"  Unique modes: {set([m for modes in df_scoring['Mode_Planned'] for m in modes])}")
print(f"  Region categories: {df_scoring['Region_category'].value_counts().to_dict()}")
print(f"  Location categories: {df_scoring['Location_category'].value_counts().to_dict()}")
print(f"  RegionLocation range: {df_scoring['RegionLocation'].min()} - {df_scoring['RegionLocation'].max()}")
print(f"  RegionLocation unique values: {sorted(df_scoring['RegionLocation'].unique())}")


## Step 4.3: Calculate Mode Score and Bus Terminal Score

In [ ]:
# Step 4.3: Calculate Mode Score and Bus Terminal Score

# MODE WEIGHTS for scoring (from CLAUDE.md)
# MODE_WEIGHTS imported from config
# Original definition (now in src/config.py):
# MODE_WEIGHTS = {
#     'Funicular': 1.0,
#     'Cable Line': 2.0,
#     'BRT': 3.0,
#     'LRT': 4.0,
#     'Metro': 5.0,
#     'Suburban Rail': 6.0,
#     'Interurban Rail': 7.0,
#     'HighSpeed Rail': 8.0,
#     'Rail': 7.0,
#     'Express Bus': 3.0,
#     'Bus': 2.0,
# }

MODE_LINE_COLS = [
    'BRT Lines', 'Cable Line Lines', 'Funicular Lines',
    'HighSpeed Rail Lines', 'Interurban Rail Lines', 'LRT Lines',
    'Metro Lines', 'Suburban Rail Lines'
]

def count_positive_mode_lines(row):
    """Count how many mode-specific line columns have values > 0."""
    count = 0
    for col in MODE_LINE_COLS:
        if col in row.index and pd.notna(row[col]) and row[col] > 0:
            count += 1
    return count

def calculate_mode_score(row):
    """Calculate mode service score with mode weights and diversity bonus."""
    score = 0.0
    alpha = 0.1  # Diversity bonus factor (10% per additional mode)

    # Calculate score for each mode
    for mode, weight in MODE_WEIGHTS.items():
        column_name = f'{mode} Lines'
        if column_name in row.index and pd.notna(row[column_name]) and row[column_name] > 0:
            # Multiply line count by mode weight
            score += row[column_name] * weight

    # Apply diversity bonus based on number of modes
    n_modes = row.get('Num_Modes', 1)
    if pd.notna(n_modes) and n_modes > 0:
        score = score * (1 + alpha * (n_modes - 1))

    return score

def bus_terminal_score(term_type):
    """Convert terminal type to score."""
    if pd.isna(term_type):
        return 0
    term_type = str(term_type).strip()
    if 'חניון לילה' in term_type:
        return 1
    elif 'מסוף קטן' in term_type or 'מסוף בינוני' in term_type:
        return 2
    elif 'מסוף גדול' in term_type or 'מתקן משולב' in term_type:
        return 3
    return 0

# Recalculate Num_Modes if not already correct
if 'Num_Modes' not in df_scoring.columns or df_scoring['Num_Modes'].max() == 0:
    print("Recalculating Num_Modes...")
    df_scoring['Num_Modes'] = df_scoring.apply(count_positive_mode_lines, axis=1)
    print(f"  Num_Modes range: {df_scoring['Num_Modes'].min()} - {df_scoring['Num_Modes'].max()}")

# Calculate mode service score
print("Calculating mode service score...")
df_scoring['score'] = df_scoring.apply(calculate_mode_score, axis=1)
print(f"  score range: {df_scoring['score'].min():.2f} - {df_scoring['score'].max():.2f}")

# Calculate bus terminal score
if 'term_type' in df_scoring.columns:
    df_scoring['bus_terminal'] = df_scoring['term_type'].apply(bus_terminal_score)
else:
    df_scoring['bus_terminal'] = 0
    print("  ⚠ No 'term_type' column, setting bus_terminal=0")

print("\n✓ Step 4.3 complete - Mode and bus terminal scores calculated!")
print(f"  Columns added/updated: Num_Modes, score, bus_terminal")
print(f"  Note: RegionLocation was already calculated in Step 4.2")


## Step 4.4: Filter Eligible Hubs and Classify Hub Types

In [ ]:
# Set tier defaults if not configured
if 'TIER_NATIONAL' not in globals():
    TIER_NATIONAL = "ארצי"
if 'TIER_METRO' not in globals():
    TIER_METRO = "מטרופוליני"
if 'TIER_LOCAL' not in globals():
    TIER_LOCAL = "עירוני"

print("Step 4.4: Filtering eligible hubs and classifying types...")

if 'df_scoring' not in globals():
    print('ERROR: Run Step 4.2 (Data cleaning) first!')
    raise NameError('df_scoring not defined. Run Step 4.2 first.')

# Filter: >= 1000 demand AND >= 2 modes
initial_count = len(df_scoring)
if 'TotalDemand' in df_scoring.columns:
    df_scoring = df_scoring[df_scoring['TotalDemand'] >= 1000].copy()
    print(f"  After demand filter (>=1000): {len(df_scoring)} hubs")

df_scoring = df_scoring[df_scoring['Mode_Planned'].apply(lambda x: len(x) > 1)].copy()
print(f"  After mode filter (>=2 modes): {len(df_scoring)} hubs")
print(f"  Filtered out: {initial_count - len(df_scoring)} hubs")

# Hub type classification
def classify_hub_tier(total_demand, modes, num_lines):
    """Classify hub based on modes, lines, and ridership."""
    if not isinstance(modes, list):
        modes = [modes] if pd.notna(modes) else []
    
    has_highspeed = 'HighSpeed Rail' in modes or 'HighSpeed' in modes
    has_interurban = 'Interurban Rail' in modes or 'Interurban' in modes
    has_suburban = 'Suburban Rail' in modes or 'Suburban' in modes
    has_metro = 'Metro' in modes
    has_brt = 'BRT' in modes
    has_lrt = 'LRT' in modes
    has_rail = 'Rail' in modes
    
    # National: (HighSpeed OR Interurban) AND >=3 lines AND >=50k demand
    if (has_highspeed or has_interurban or (has_rail and num_lines >= 3)) and (num_lines >= 3) and (total_demand >= 50000):
        return TIER_NATIONAL
    # Regional: (Suburban OR Metro OR Interurban OR HighSpeed) AND >=3 lines
    elif (has_suburban or has_metro or has_interurban or has_highspeed or has_rail) and (num_lines >= 3):
        return TIER_METRO
    # Local: (BRT OR LRT) AND >=3 lines AND >=1000 demand
    elif (has_brt or has_lrt) and (num_lines >= 3) and (total_demand >= 1000):
        return TIER_LOCAL
    # Train Station: rail modes but <=2 lines
    elif (has_interurban or has_highspeed or has_suburban or has_rail) and (num_lines <= 2):
        return 'Train Station'
    else:
        return 'Not Hub'

df_scoring['HubType'] = df_scoring.apply(
    lambda row: classify_hub_tier(
        total_demand=row.get('TotalDemand', 0),
        modes=row['Mode_Planned'],
        num_lines=row['Total_Unique_Lines']
    ),
    axis=1
)

print(f"\n✓ Hub types classified:")
print(df_scoring['HubType'].value_counts().to_string())

### Optional: Non-Rail Transit Mode Filtering

**Configuration**: Set `REQUIRE_NON_RAIL_MODE = True` in `src/config.py` to enable this filter.

When enabled, hubs must have at least one non-rail transit mode (Metro, LRT, or BRT) to be eligible. This excludes "rail-only" hubs that have combinations of:
- Suburban Rail
- Interurban Rail  
- HighSpeed Rail
- Generic Rail

**Rationale**: True multimodal hubs should integrate urban transit (Metro/LRT/BRT) with rail, not just rail-to-rail transfers.


In [ ]:
# Non-rail transit mode filtering
# This runs automatically if REQUIRE_NON_RAIL_MODE = True in config

if REQUIRE_NON_RAIL_MODE:
    print(f"Non-rail mode filtering: ENABLED")
    print(f"  Rail-only modes: {RAIL_ONLY_MODES}")
    print(f"  Non-rail transit modes: {NON_RAIL_TRANSIT_MODES}")
    
    # Check each hub for rail-only status
    def is_rail_only(modes_list):
        """Check if hub has only rail modes (no Metro, LRT, BRT)."""
        if not modes_list:
            return False
        return all(mode in RAIL_ONLY_MODES for mode in modes_list if mode is not None)
    
    # Apply to df_scoring from Step 4.4
    if 'df_scoring' in globals():
        initial_count = len(df_scoring)
        
        # Apply non-rail filter
        df_scoring['is_rail_only'] = df_scoring['Mode_Planned'].apply(is_rail_only)
        rail_only_count = df_scoring['is_rail_only'].sum()
        
        df_scoring = df_scoring[~df_scoring['is_rail_only']].copy()
        df_scoring = df_scoring.drop(columns=['is_rail_only'])
        
        print(f"  ✓ Filtered out {rail_only_count} rail-only hubs")
        print(f"  ✓ Remaining: {len(df_scoring)}/{initial_count} hubs")
    else:
        print("  ⚠ df_scoring not found - run filtering steps first")
else:
    print(f"Non-rail mode filtering: DISABLED (REQUIRE_NON_RAIL_MODE = {REQUIRE_NON_RAIL_MODE})")
    print(f"  All hubs with 2+ mass-transit modes are eligible, regardless of mode mix")


## Step 4.5: Normalize Scores and Calculate Pop/Emp Score

In [ ]:
# Set distance decay default if not configured
if 'DISTANCE_DECAY_BETA' not in globals():
    DISTANCE_DECAY_BETA = 1.5

print("Step 4.5: Normalizing scores and calculating Pop/Emp score...")

if 'df_scoring' not in globals():
    print('ERROR: Run Step 4.2 (Data cleaning) first!')
    raise NameError('df_scoring not defined. Run Step 4.2 first.')

# Normalization function
def normalize_col_by_type(df, col, hub_type_col='HubType'):
    """Normalize to 1-10 scale per hub type."""
    normalized_col = col + '_Norm'
    for hub_type in df[hub_type_col].unique():
        mask = df[hub_type_col] == hub_type
        values = df.loc[mask, col]
        min_val, max_val = values.min(), values.max()
        if max_val > min_val:
            df.loc[mask, normalized_col] = 1 + (values - min_val) * 9 / (max_val - min_val)
        else:
            df.loc[mask, normalized_col] = 5.5
    return df

# Normalize categorical scores
for col in ['RegionLocation', 'score', 'bus_terminal']:
    df_scoring = normalize_col_by_type(df_scoring, col)

# Normalize demand with log10
if 'TotalDemand' in df_scoring.columns:
    df_scoring['LogDemand'] = df_scoring['TotalDemand'].apply(lambda x: 0 if x == 0 else np.log10(x))
    for hub_type in df_scoring['HubType'].unique():
        mask = df_scoring['HubType'] == hub_type
        values = df_scoring.loc[mask, 'LogDemand']
        values_nonzero = values[values > 0]
        if len(values_nonzero) > 0:
            min_val, max_val = values_nonzero.min(), values_nonzero.max()
            if max_val > min_val:
                df_scoring.loc[mask, 'TotalDemand_Norm'] = 1 + (values - min_val) * 9 / (max_val - min_val)
            else:
                df_scoring.loc[mask, 'TotalDemand_Norm'] = 5.5
        else:
            df_scoring.loc[mask, 'TotalDemand_Norm'] = 1.0

# Pop/Emp score with distance decay
pop_cols = ['pop_0_500', 'pop_500_1000', 'pop_1000_1500']
emp_cols = ['emp_0_500', 'emp_500_1000', 'emp_1000_1500']

if all(col in df_scoring.columns for col in pop_cols + emp_cols):
    radii = [0, 500, 1000, 1500]
    mids = np.array([(radii[i] + radii[i+1]) / 2 for i in range(len(radii)-1)])
    decay = mids ** DISTANCE_DECAY_BETA
    
    scores = []
    for _, row in df_scoring.iterrows():
        pop = np.array([row.get(col, 0) for col in pop_cols], dtype=float)
        emp = np.array([row.get(col, 0) for col in emp_cols], dtype=float)
        
        # Weight by hub type
        if row['HubType'] in ['National', 'Regional', TIER_NATIONAL, TIER_METRO]:
            w_pop, w_emp = 0.2, 0.8
        else:
            w_pop, w_emp = 0.8, 0.2
        
        combined = w_pop * pop + w_emp * emp
        score = (combined / decay).sum()
        scores.append(score)
    
    df_scoring['PopEmp_Score_Raw'] = scores
    
    # Normalize per hub type
    for hub_type in df_scoring['HubType'].unique():
        mask = df_scoring['HubType'] == hub_type
        values = df_scoring.loc[mask, 'PopEmp_Score_Raw']
        min_val, max_val = values.min(), values.max()
        if max_val > min_val:
            df_scoring.loc[mask, 'PopEmp_Score_Norm'] = 1 + (values - min_val) * 9 / (max_val - min_val)
        else:
            df_scoring.loc[mask, 'PopEmp_Score_Norm'] = 5.5
else:
    print("  ⚠ Pop/Emp columns not found, using default score")
    df_scoring['PopEmp_Score_Norm'] = 5.0

print(f"✓ Scores normalized")
for col in ['RegionLocation_Norm', 'score_Norm', 'bus_terminal_Norm', 'TotalDemand_Norm', 'PopEmp_Score_Norm']:
    if col in df_scoring.columns:
        print(f"  {col}: {df_scoring[col].min():.2f} - {df_scoring[col].max():.2f}")

## Step 4.6: Monte Carlo Aggregation and Rankings

In [ ]:
# Set defaults if not already configured
# Check prerequisites
if 'INPUT_FOR_SCORING' not in globals():
    print('\n' + '='*80)
    print('ERROR: INPUT_FOR_SCORING not found!')
    print('='*80)
    print('\nYou need to run Step 4.1 (Configuration) first,')
    print('which sets INPUT_FOR_SCORING based on Parts 2/3 output.')
    print('\nMake sure you have run:')
    print('  - Part 1 (H3 hexagons)')
    print('  - Part 2 (Demand data) - recommended')
    print('  - Part 3 (Demographics) - optional')
    print('  - Step 4.1 (Part 4 configuration)')
    print('\nThen run this cell again.\n')
    raise NameError('INPUT_FOR_SCORING not defined. Run Step 4.1 first.')

if 'RANDOM_SEED' not in globals():
    RANDOM_SEED = MONTE_CARLO_RANDOM_SEED  # From config
if 'TIER_NATIONAL' not in globals():
    TIER_NATIONAL = "ארצי"
if 'TIER_METRO' not in globals():
    TIER_METRO = "מטרופוליני"
if 'TIER_LOCAL' not in globals():
    TIER_LOCAL = "עירוני"

print(f"Step 4.6: Running Monte Carlo simulation ({MONTE_CARLO_ITERATIONS:,} iterations)...")

# Check prerequisites
if 'df_scoring' not in globals():
    print('\n' + '='*80)
    print('ERROR: df_scoring not found!')
    print('='*80)
    print('\nYou need to run the previous Part 4 cells first:')
    print('  1. Step 4.1: Configuration (optional, has defaults)')
    print('  2. Step 4.2: Data cleaning and preparation')
    print('  3. Step 4.3: Mode score calculation')
    print('  4. Step 4.4: Hub type classification')
    print('  5. Step 4.5: Score normalization')
    print('  6. Then run this cell (Step 4.6)')
    print('\nPlease run these cells in order.\n')
    raise NameError('df_scoring is not defined. Run previous Part 4 cells first.')

scoring_cols = ['RegionLocation_Norm', 'bus_terminal_Norm', 'score_Norm', 
                'TotalDemand_Norm', 'PopEmp_Score_Norm']

# Check all columns exist
missing = [col for col in scoring_cols if col not in df_scoring.columns]
if missing:
    print(f"  ⚠ Missing score columns: {missing}")
    print(f"  Skipping Monte Carlo")
    df_scoring['Average_Simulated_Score'] = 5.0
else:
    np.random.seed(RANDOM_SEED)
    results = []
    
    for hub_type in df_scoring['HubType'].unique():
        hub_df = df_scoring[df_scoring['HubType'] == hub_type].copy()
        hub_df['Sum_Simulated_Scores'] = 0.0
        
        for i in range(MONTE_CARLO_ITERATIONS):
            # Generate random weights (max 0.5 per criterion)
            weights = np.random.rand(len(scoring_cols))
            while any(weights > 0.5):
                weights = np.random.rand(len(scoring_cols))
            weights /= weights.sum()
            
            # Calculate weighted scores
            hub_df['Sum_Simulated_Scores'] += (hub_df[scoring_cols] * weights).sum(axis=1)
        
        # Average across iterations
        hub_df['Average_Simulated_Score'] = hub_df['Sum_Simulated_Scores'] / MONTE_CARLO_ITERATIONS
        hub_df = hub_df.drop(columns='Sum_Simulated_Scores')
        results.append(hub_df)
    
    df_scored = pd.concat(results, ignore_index=False).sort_index()
    
    # Add rankings
    df_scored['Rank_within_HubType'] = df_scored.groupby('HubType')['Average_Simulated_Score'].rank(
        method='dense', ascending=False
    )
    df_scored['Overall_Rank'] = df_scored['Average_Simulated_Score'].rank(
        method='dense', ascending=False
    )
    
    print(f"\n✓ Monte Carlo complete!")
    print(f"  Final scores: {df_scored['Average_Simulated_Score'].min():.3f} - {df_scored['Average_Simulated_Score'].max():.3f}")
    print(f"  Mean: {df_scored['Average_Simulated_Score'].mean():.3f}")
    
    print(f"\n  Top 10 hubs overall:")
    top_10 = df_scored.nlargest(10, 'Average_Simulated_Score')
    for i, (idx, row) in enumerate(top_10.iterrows(), 1):
        hub_id = row.get('group', idx)
        score = row['Average_Simulated_Score']
        hub_type = row['HubType']
        demand = row.get('TotalDemand', 0)
        print(f"    {i}. Hub {hub_id} ({hub_type}): {score:.3f} | Demand: {demand:,.0f}")
    
    # Summary by hub type
    print(f"\n  Results by Hub Type:")
    for hub_type in df_scored['HubType'].unique():
        hub_data = df_scored[df_scored['HubType'] == hub_type]
        print(f"    {hub_type}: {len(hub_data)} hubs, scores {hub_data['Average_Simulated_Score'].min():.3f}-{hub_data['Average_Simulated_Score'].max():.3f}")

## Step 4.7: Export Scored and Ranked Hubs

In [ ]:
if 'df_scored' in globals():
    print("Step 4.7: Exporting scored and ranked hubs...")
    print("="*80)

    # Prepare final output dataframe
    df_export = df_scored.copy()

    # Add centroid, x, y columns if not already present
    if 'geometry' in df_export.columns and 'centroid' not in df_export.columns:
        print("  Creating centroid, x, and y columns...")
        # Create centroid from geometry
        if hasattr(df_export, 'geometry'):
            df_export['centroid'] = df_export.geometry.centroid
            df_export['x'] = df_export.centroid.x
            df_export['y'] = df_export.centroid.y
            print(f"  ✓ Added centroid (x, y) columns")
    else:
        print("  Centroid columns already present or no geometry column")

    # Define columns to export - using actual column names from pipeline
    export_cols = [
        # Core identification
        'group',              # Hub ID (group number from clustering)
        'x', 'y',             # Coordinates
        
        # Hub classification
        'HubType',            # Hub tier (ארצי/מטרופוליני/עירוני/תחנת רכבת)
        
        # Demand data
        'TotalDemand',        # Total daily demand
        'TotalTransfers',     # Transfer passengers
        
        # Mode information
        'Mode_Planned',       # List of planned modes
        'Num_Modes',          # Number of different modes
        'Line_Nunique',       # Number of unique lines
        
        # Location data
        'area',               # Geographic area (metro region)
        'location',           # Metropolitan position (core/ring)
        'Region_category',    # Region category (0=center, 1=periphery)
        'Location_category',  # Location category (1-3)
        
        # Raw scores (before normalization)
        'RegionLocation',     # Region × Location score
        'score',              # Mode service score
        'bus_terminal',       # Bus terminal proximity score
        'PopEmp_Score',       # Population/Employment score
        
        # Normalized scores (1-10 scale, per hub type)
        'RegionLocation_Norm',
        'score_Norm',
        'bus_terminal_Norm',
        'TotalDemand_Norm',
        'PopEmp_Score_Norm',
        
        # Final Monte Carlo results
        'Average_Simulated_Score',  # Final aggregated score
        'Overall_Rank',             # Overall ranking
        'Rank_within_HubType',      # Ranking within hub type
        
        # Original data
        'h3_index',           # H3 hexagon indices
        'node',               # Node IDs
        'address',            # Address (if geocoded)
    ]

    # Add mode-specific line count columns if they exist
    mode_line_cols = [
        'BRT Lines', 'Cable Line Lines', 'Funicular Lines',
        'HighSpeed Rail Lines', 'Interurban Rail Lines', 'LRT Lines',
        'Metro Lines', 'Suburban Rail Lines'
    ]
    export_cols.extend(mode_line_cols)

    # Population/Employment zone columns
    pop_emp_cols = [
        'pop_0_500', 'pop_500_1000', 'pop_1000_1500',
        'emp_0_500', 'emp_500_1000', 'emp_1000_1500',
        'pop_zone1', 'pop_zone2', 'pop_zone3',
        'emp_zone1', 'emp_zone2', 'emp_zone3',
    ]
    export_cols.extend(pop_emp_cols)

    # Only include columns that exist in the dataframe
    export_cols = [c for c in export_cols if c in df_export.columns]
    
    print(f"\n  Columns to export ({len(export_cols)}):")
    for col in export_cols:
        print(f"    - {col}")

    # Export to CSV
    print(f"\n  Exporting {len(df_export)} hubs to CSV...")
    df_export[export_cols].to_csv(OUTPUT_SCORED_HUBS, index=False, encoding='utf-8-sig')
    print(f"  ✓ CSV exported: {OUTPUT_SCORED_HUBS}")

    # Export to Excel with formatting
    print(f"\n  Exporting to Excel with formatting...")
    with pd.ExcelWriter(OUTPUT_SCORED_EXCEL, engine='openpyxl') as writer:
        df_export[export_cols].to_excel(writer, sheet_name='Scored Hubs', index=False)

        # Get workbook and worksheet
        worksheet = writer.sheets['Scored Hubs']

        # Auto-adjust column widths (handle columns beyond 'Z')
        for idx, col in enumerate(export_cols, 1):
            max_length = max(
                df_export[col].astype(str).apply(len).max(),
                len(col)
            ) + 2
            # Handle column letters (A-Z, AA-AZ, etc.)
            if idx <= 26:
                col_letter = chr(64 + idx)
            else:
                col_letter = chr(64 + (idx - 1) // 26) + chr(65 + (idx - 1) % 26)
            worksheet.column_dimensions[col_letter].width = min(max_length, 50)

    print(f"  ✓ Excel exported: {OUTPUT_SCORED_EXCEL}")

    # Summary
    print(f"\n  Summary:")
    print(f"    Total hubs: {len(df_export)}")
    print(f"    Columns exported: {len(export_cols)}")

    # Show score statistics
    if 'Average_Simulated_Score' in df_export.columns:
        print(f"\n  Score Statistics:")
        print(f"    Min: {df_export['Average_Simulated_Score'].min():.3f}")
        print(f"    Max: {df_export['Average_Simulated_Score'].max():.3f}")
        print(f"    Mean: {df_export['Average_Simulated_Score'].mean():.3f}")

    # Show hub type distribution
    if 'HubType' in df_export.columns:
        print(f"\n  Hub Type Distribution:")
        for hub_type, count in df_export['HubType'].value_counts().items():
            print(f"    {hub_type}: {count}")

    print(f"\n  ✓ Export complete!")
    print(f"    Output CSV: {OUTPUT_SCORED_HUBS}")
    print(f"    Output Excel: {OUTPUT_SCORED_EXCEL}")
else:
    print("⚠ df_scored not found - run Step 4.6 (Monte Carlo) first")
